# <center>容器化部署:Docker + Compose</center>

&emsp;&emsp;在《Agent 智能体部署上线》的课程中,我们已经用 systemd + nginx + certbot + venv 这一套手动方式把一个智能体项目通过 HTTPS 跑在腾讯云 VPS 上。这条路走通了,但有个问题始终没解决:**换一台机器,这一整套部署流程要从头再来一遍**。本门课就是来解决这件事——把"手动装服务"换成"分发一份镜像",把"一台机器"换成"任何机器"。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/0.0-%E8%AF%BE%E7%A8%8B%E5%BC%80%E7%AF%87-%E4%BB%8E%E5%8D%95%E5%AE%B9%E5%99%A8%E5%88%B0%E5%A4%9A%E5%AE%B9%E5%99%A8-09fbce08.png" width=80%>
</div>

&emsp;&emsp;本课程的主要路径是<b>从单容器到多容器</b>。前半段(第三、四章)我们使用一个<b>单进程智能体项目</b>(Python Web 服务 + SQLite 数据库)学单容器——用 Dockerfile 把它打包成镜像 → 推 Docker Hub → 腾讯云 CVM(amd64 通用云)拉下跑起,讲清楚"打包一个进程"。后半段(第五、六章)使用<b>多服务的文搜图项目</b>——FastAPI + Qwen3-VL 双模型 + Milvus 三件套 + Vite 前端共 5 服务互相协作,一份 `compose.yml` 把"服务间依赖 / 内部网络 / 卷挂载 / 健康检查"打包成一条命令,讲清楚"编排一组进程",最终在<b>企业服务器</b>(Ubuntu 22.04 + 4× RTX 3090 + Docker 29.1.3 + nvidia CDI,GPU 推理需要)上跑起来。单容器与多容器的部署目标差异也是工业部署的真实分工:单进程用通用云足够,GPU 推理 + 多服务编排走企业自有 GPU 服务器。七章按"零基础认 Docker → 单容器 → 跨架构推 Hub + 云上拉 → 多容器 compose 编排 → 企业服务器“4GPU24G“上线 → 回顾"递进,学完带走 6 件产物。

---

## <center>第一章 Docker 是什么:镜像、容器、仓库三对象</center>

&emsp;&emsp;本章从零认识 Docker——不是"docker 命令教程",而是把<b>镜像 / 容器 / 仓库</b>这三个对象的关系先理清,所以全章不抛一行 docker 命令(命令留到第二章)。1.1 节先讲 Docker 是什么 + 没有它之前我们怎么做,1.2-1.5 分别展开镜像 / 容器 / 仓库 + 容器 vs 虚拟机的边界,把三对象的关系理清再进入第二章的工具实操。

### 1.1 Docker 是什么

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/1.1-Docker%E6%98%AF%E4%BB%80%E4%B9%88-ec8120af.png" width=80%>
</div>

&emsp;&emsp;<b>Docker 是一个用于开发、交付和运行应用程序的开放平台。</b>

&emsp;&emsp;为了让你更直观地理解这个官方定义，业界最经典的类比就是集装箱 (Shipping Container)：在集装箱发明之前，运输货物极其痛苦：钢琴怕水、玻璃怕碎、生鲜怕热，码头工人（运维人员）面对不同的货物（软件应用）需要定制不同的搬运方案（配置环境）。

&emsp;&emsp;<b>Docker</b> 就是软件世界的集装箱标准。它不在乎你这个“盒子”里装的是 Python、Java 还是 Go 代码，也不在乎底层承载它的是哪种操作系统的货轮。只要你打包成了标准尺寸的集装箱（镜像），底层的操作系统只需要提供一台标准化的起重机（Docker Engine），就能把它完美地卸货并运行起来。

——它把"代码 + 解释器 + 依赖 + 配置"打包成一份标准化的<b>镜像</b>(image),任何装了 Docker 的机器拉到这份镜像就能跑出隔离、行为一致的<b>容器</b>(container)。一句话总结:<b>用一份镜像替代手装环境</b>。整套生态的核心三对象是<b>镜像 / 容器 / 仓库</b>——镜像是 build 出来的不可变模板,容器是镜像跑起来的运行实例,仓库是镜像的存放与分发处。

&emsp;&emsp;本章的诉求一句话:<b>把"装服务"换成"分发一份镜像"</b>。我们希望这份镜像无论上腾讯云、上同事电脑、上另一家云厂商,跑起来都是一致的——同样的 Python 版本、同样的依赖、同样的目录结构、同样的启动命令。Docker 解决的就是这件事。

### 1.2 镜像(Image):创建容器的只读模板

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/1.2-%E9%95%9C%E5%83%8F%E5%88%86%E5%B1%82%E7%BB%93%E6%9E%84-a9cf9abf.png" width=80%>
</div>

&emsp;&emsp;镜像(Image)是一份装好所有运行环境的标准化产物:代码、解释器、依赖、配置文件全部就位,以分层只读结构存放。

&emsp;&emsp;有了这份镜像,"换机器还能跑"这件事就被换成了"换机器拉一份镜像"。任何一台机器,只要装了 Docker,拉到这份镜像就能用——不再依赖目标机器装没装过 Python、apt 源是否能连、Python 版本几点几这些环境差异。

&emsp;&emsp;镜像不是简单压缩包。镜像内部按层(layer)组织,每层是一次文件系统变更的差分(改了哪些文件、加了哪些目录)。这层结构在第三章 Dockerfile 构建时会展开——现在先记住"镜像是不可变制品"就够了。

&emsp;&emsp;另一件需要先有个印象的事:**镜像通常不是从零写的,而是基于一份<b>基础镜像(base image)</b>叠加自己的代码层**——Python 项目从 `python:3.11-slim` 开始,Web 项目从 `nginx:alpine` 开始,基础镜像本身也是别人 build 好上传到 Docker Hub 的镜像。这种"在别人的镜像上叠自己的层"的复用结构带来两个红利:① <b>build 速度极快</b>(基础镜像 layer 命中缓存就跳过)、② <b>镜像体积可控</b>(不同镜像共享底层 base layer)。第三章我们写第一份 Dockerfile 时,第一行 `FROM python:3.12-slim` 就是声明基础镜像。

> <font size=2>**【名词解释】<font color=red>基础镜像</font>**(base image,镜像构建的起点)— Dockerfile 第一行 `FROM xxx` 声明的那份镜像,后续所有指令在它之上叠加自己的层。基础镜像通常由官方或社区在 Docker Hub 上维护,体积小、装好基础运行时、安全更新及时。常见基础镜像分四类:<br>
> ① <b>语言运行时类</b>:`python:3.11-slim`(Python + pip,~80 MB)、`node:20-slim`(Node + npm)、`openjdk:21`(Java)、`golang:1.22`(Go)、`ruby:3.3`<br>
> ② <b>Web 服务类</b>:`nginx:alpine`(nginx + Alpine Linux,~40 MB)、`caddy:2`(自动 HTTPS)、`httpd:2.4`(Apache)<br>
> ③ <b>数据库 / 中间件类</b>:`postgres:16`、`mysql:8`、`redis:7-alpine`、`mongo:7`、`rabbitmq:3-management`<br>
> ④ <b>操作系统底座</b>:`ubuntu:22.04`(完整 Ubuntu)、`debian:12-slim`、`alpine:3.19`(最小基础,~5 MB,自带 musl libc + apk)<br>
> 同一基础镜像往往有多个 tag 变体:`-slim`(去掉文档和不常用工具,体积更小)、`-alpine`(基于 Alpine Linux,极小)、不加后缀的完整版(自带 `build-essential` 等编译工具,适合多阶段构建的 builder 阶段)。本课第三章 ai-todo 用 `python:3.12-slim`、第五章 backend 多阶段构建用 `python:3.11`(builder)+ `python:3.11-slim`(runtime)。</font>

> <font size=2>**【名词解释】<font color=red>Dockerfile</font>(Dockerfile,镜像构建脚本)** — Docker 官方定义是"一份文本文档,包含用户从命令行能调用的所有指令,用来组装一个镜像",内含 `FROM` / `COPY` / `RUN` / `CMD` 等指令。</font>

### 1.3 容器(Container):镜像的运行时实例

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/1.3-%E5%AE%B9%E5%99%A8%E6%98%AF%E9%95%9C%E5%83%8F%E8%BF%90%E8%A1%8C%E5%AE%9E%E4%BE%8B-f5edb24e.png" width=80%>
</div>

&emsp;&emsp;容器(Container)是镜像跑起来的一个独立进程实例,有自己的文件系统视图、自己的网络命名空间、自己的进程空间,但和宿主机共用内核。

&emsp;&emsp;一份镜像可以同时跑出多份容器,互不干扰:上午跑一个监听 12234 端口的开发副本、下午另起一个监听 12235 端口的测试副本,两份容器各自独立、不共享内存、不共享数据库文件。容器停掉再启动,代码和依赖永远是镜像里的那一份——不会因为"上次跑的时候改了某个文件"导致下次启动行为不一致。

&emsp;&emsp;容器删掉之后,容器内写过的临时文件会一起消失。需要持久化的数据(比如 Web 服务的 SQLite 数据库文件)要靠"卷"留住,这点在第三章会展开。

### 1.4 容器 vs 虚拟机 vs 物理服务器

&emsp;&emsp;初次接触容器时很容易冒出一个问题：这和虚拟机有什么区别？和直接在服务器上跑进程又有什么区别？

&emsp;&emsp;<b>物理服务器（裸机）</b>是最直接的方式：一台机器跑一个操作系统，进程直接跑在这个 OS 上，所有进程共用同一套内核和文件系统。传统的云 VPS 部署就是这个模式——Web 服务进程和 nginx 进程都跑在同一个 Linux 系统里，没有任何隔离，一个进程乱写文件可能波及另一个。

&emsp;&emsp;<b>虚拟机（VM）</b>在物理机上面加了一层 Hypervisor，虚拟出完整的虚拟硬件，每台 VM 有自己独立的操作系统内核。隔离是最彻底的——A VM 里的进程崩了完全不影响 B VM。代价是每台 VM 都要跑一个完整的 OS，内存动辄几个 GB，启动要几十秒甚至几分钟。

&emsp;&emsp;<b>容器</b>走了另一条路：不虚拟硬件、不各自跑一个 OS 内核，而是所有容器共用宿主机的 Linux 内核，通过 Linux 的 <b>namespace</b>（命名空间）和 <b>cgroup</b>（资源控制）这两个内核机制，让每个容器感觉自己有独立的文件系统、网络和进程空间。<b>隔离强度处于裸机和 VM 之间</b>——比 VM 轻得多(没有独立 OS 内核 + 不模拟硬件),但比裸机进程多了一层 namespace + cgroup 把文件系统 / 网络 / 进程空间隔开。代价是镜像只有几十 MB,启动不到一秒。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/1.4-%E5%AE%B9%E5%99%A8%E5%AF%B9%E6%AF%94%E8%99%9A%E6%8B%9F%E6%9C%BA-3934fefb.png" width=80%>
</div>

&emsp;&emsp;一句话记住核心差异：<b>VM 模拟整台电脑，容器只隔离进程</b>。容器能做到"任何机器跑起来一致"靠的是镜像把依赖锁住，不是靠虚拟出一套硬件。这也解释了为什么 Docker Desktop 在 macOS 和 Windows 上仍然需要一个轻量 Linux 虚拟机——容器依赖 Linux 内核机制，macOS 和 Windows 本身没有，所以 Docker Desktop 在底层偷偷跑了一个 Linux VM 来承载容器，我们操作的命令只是控制这个 VM 里的 Docker daemon。

### 1.5 仓库(Registry):镜像的存储与分发服务

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/1.5-Registry%E5%AD%98%E5%82%A8%E4%B8%8E%E5%88%86%E5%8F%91-97d5b015.png" width=80%>
</div>

&emsp;&emsp;仓库(Registry)是存放镜像的远程服务,最常用的公开仓库是 Docker Hub(域名 `docker.io`),云厂商也都有自家镜像仓库(腾讯云镜像仓库地址 `ccr.ccs.tencentyun.com`、阿里云 `registry.cn-hangzhou.aliyuncs.com`、华为云 `swr.cn-north-4.myhuaweicloud.com`)。

&emsp;&emsp;本机构建出来的镜像 push 上去,腾讯云的机器 pull 下来就能 run——这是本课"打包 + 分发"叙事的核心。没有仓库这一环,镜像就只能停在本机,所谓"换机器还能跑"也就无从谈起。

---

## <center>第二章 Docker Desktop 上手</center>

&emsp;&emsp;第一章把三个对象的关系讲清楚了,但还没在工具里看到它们的实际样子。<b>Docker Desktop 是 Docker 公司提供的桌面应用</b>——把 <b>Docker Engine</b>(执行容器的后台 daemon + `docker` CLI)+ GUI 管理界面 + 必要的 Linux 虚拟机(macOS / Windows 上跑 Linux 容器要的,因为容器依赖 Linux 内核机制)打包成开箱即用的安装包,装上就能在 macOS / Windows / Linux 三平台跑 `docker` 命令。本课所有"本机操作"都在 Docker Desktop 里完成;到了第四章上腾讯云、第六章上 4GPU24G 服务器,服务器端跑的是纯 <b>Docker Engine</b>(没 GUI 也不需要虚拟机,因为本身就是 Linux),CLI 命令完全一致。本章把 Docker Desktop 装好,在界面里把镜像 / 容器找出来,跑第一个 hello-world 体会 `docker pull → docker run → docker ps → docker rm/rmi` 的最小闭环。这一章不讲项目,纯粹是工具熟悉。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/2.0-Docker-Desktop%E6%80%BB%E8%A7%88-06a5fbff.png" width=80%>
</div>

### 2.1 三平台安装

&emsp;&emsp;Docker Desktop 在 macOS / Linux / Windows 都提供桌面应用。我们按三平台分别给出安装命令,装完打开应用看到鲸鱼图标常驻在系统状态栏即为成功。

```bash
# macOS:推荐用 Homebrew,自动处理 Apple Silicon / Intel 架构差异
brew install --cask docker
open -a Docker
```

```bash
# Linux(Ubuntu / Debian)下载 deb 包,sudo apt 装本地包
curl -fsSL -o docker-desktop.deb \
    https://desktop.docker.com/linux/main/amd64/docker-desktop-amd64.deb
sudo apt install -y ./docker-desktop.deb
systemctl --user start docker-desktop
```

```powershell
# Windows PowerShell:用 winget 拉官方 Installer
# 装完会在开始菜单出现 Docker Desktop 应用,首次启动会要求选 WSL 2 backend
winget install Docker.DockerDesktop
```

&emsp;&emsp;上面这三段命令分别覆盖三平台的应用安装。把 Docker Desktop 桌面应用装到本机,包含 docker daemon(后台进程)、docker CLI、buildx、Compose 子命令。本课所有本机操作都通过 Docker Desktop,装好是后续一切命令的前提。容器化部署不区分前端开发还是后端运维,装一份 Docker Desktop 之后,镜像 / 容器 / 卷 / 网络都在同一份工具里统一管理。

> <font size=2>**【名词解释】<font color=red>docker daemon</font>(docker daemon / dockerd,Docker 后台进程)** — Docker 官方定义是"创建和管理镜像、容器、网络、卷等 Docker 对象"的服务端长驻进程。CLI 和 GUI 都是它的客户端,Docker Desktop 启动后这个进程在一个轻量 Linux 虚拟机里跑。</font>

&emsp;&emsp;Linux 这边需要补充一点:Linux 也可以选择装更轻量的 <b>Docker Engine</b>(只有 daemon + CLI,没有 GUI),装法是 `apt install docker-ce`。

### 2.2 配置 docker mirror

&emsp;&emsp;<b>为什么要配 mirror</b>:Docker Desktop 装完默认走 Docker Hub(`registry-1.docker.io`)直连拉镜像,这是个**架在美国的全球公开仓库**——国内网络到它的链路又慢又抖,加上 Hub 对匿名拉取限额很严(每 6 小时 100 次,第四章 4.6 节会展开),实测 `docker pull python:3.12-slim` 这种 50 MB 基础镜像本机直连经常卡到几分钟或者直接超时。<b>Mirror 是 Docker Hub 的国内透明代理</b>——把同样的镜像内容缓存在国内 CDN 节点,从国内拉极快(几秒到几十秒),还绕开了 Hub 的匿名限额。所以装完 Docker Desktop 立刻配一下 mirror,后面第三、四、五章 build / pull 镜像就都顺。三家通用公网 mirror 长期可用——`docker.m.daocloud.io`(DaoCloud) / `docker.1ms.run` / `dockerhub.icu`,Mac/Linux/Windows 都能用。三平台 `daemon.json` 路径:

<style>.center{width:auto;display:table;margin-left:auto;margin-right:auto;}</style>
<p align="center"><font face="黑体" size=4>三平台 daemon.json 路径速查</font></p>
<div class="center">

| 平台 | daemon.json 路径 | 修改方式 |
|---|---|---|
| macOS | `~/.docker/daemon.json` | 直接编辑 或 Docker Desktop Settings → Docker Engine JSON 框 |
| Linux | `/etc/docker/daemon.json` | 直接编辑(需 sudo) |
| Windows | `$env:USERPROFILE\.docker\daemon.json` (PowerShell) 或 `%USERPROFILE%\.docker\daemon.json` (cmd) | 直接编辑 或 Docker Desktop Settings → Docker Engine JSON 框 |

</div>

&emsp;&emsp;最方便的入口是 Docker Desktop Settings → Docker Engine 的 JSON 编辑框。在原有 JSON 里新增 `registry-mirrors` 字段,其他字段保留:

```json
{
  "builder": {
    "gc": {
      "defaultKeepStorage": "20GB",
      "enabled": true
    }
  },
  "experimental": false,
  "registry-mirrors": [
    "https://docker.m.daocloud.io",
    "https://docker.1ms.run",
    "https://dockerhub.icu"
  ]
}
```

&emsp;&emsp;`registry-mirrors` 按数组顺序依次尝试,三家轮转给冗余。`builder` 是 Docker Desktop 自带的构建缓存 GC 配置,保留。<b>最方便的路径还是 Docker Desktop GUI</b>(Settings → Docker Engine → JSON 编辑框 → Apply & Restart)。如果你习惯命令行,<b>每个平台一条命令直接复制粘贴</b>就能写好 daemon.json:

```bash
# === macOS — 一条命令写完 daemon.json, 然后顶栏菜单 → Restart Docker Desktop
mkdir -p ~/.docker && cat > ~/.docker/daemon.json <<'EOF'
{
  "builder": {
    "gc": {"defaultKeepStorage": "20GB", "enabled": true}
  },
  "experimental": false,
  "registry-mirrors": [
    "https://docker.m.daocloud.io",
    "https://docker.1ms.run",
    "https://dockerhub.icu"
  ]
}
EOF
```

```bash
# === Linux — 一条命令写完 daemon.json + 重启 daemon (纯 Docker Engine 场景)
sudo mkdir -p /etc/docker && sudo tee /etc/docker/daemon.json > /dev/null <<'EOF' && sudo systemctl restart docker
{
  "registry-mirrors": [
    "https://docker.m.daocloud.io",
    "https://docker.1ms.run",
    "https://dockerhub.icu"
  ]
}
EOF
```

```powershell
# === Windows PowerShell — 一条命令写完 daemon.json, 然后顶栏菜单 → Restart Docker Desktop
New-Item -ItemType Directory -Force "$env:USERPROFILE\.docker" | Out-Null; @'
{
  "builder": {
    "gc": {"defaultKeepStorage": "20GB", "enabled": true}
  },
  "experimental": false,
  "registry-mirrors": [
    "https://docker.m.daocloud.io",
    "https://docker.1ms.run",
    "https://dockerhub.icu"
  ]
}
'@ | Set-Content "$env:USERPROFILE\.docker\daemon.json"
```

&emsp;&emsp;改完用 `docker info` 验证 mirror 已注册到位:

```bash
# macOS / Linux / Git Bash
docker info 2>&1 | grep -A 5 "Registry Mirrors"
# 期望: Registry Mirrors: 下面列出三家 URL
```

```powershell
# Windows PowerShell - grep 不可用, 用 Select-String -Context
docker info | Select-String "Registry Mirrors" -Context 0,5
```

> 提示:mirror 是<b>透明代理</b>——`docker pull yanggggg/wensoutu-backend:0.1` 不用改名,daemon 自动走 mirror。**只对公开仓库透明**,Hub 真私有镜像仍要 `docker login` 鉴权。

### 2.3 Docker Desktop重要模块

&emsp;&emsp;装好之后打开 Docker Desktop,左侧导航栏列出几个常用模块,跟本课直接相关的有 4 个:Containers(容器)、Images(镜像)、Volumes(卷)、Builds(构建历史)。每个模块的职责一句话说清,再对照命令行同一件事的写法。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Docker Desktop GUI 重要模块职责一览</font></p>
<div class="center">

| 模块 | 看什么 | 对应命令行 |
|---|---|---|
| Containers | 正在跑 + 已停的容器(状态 / 端口 / 资源占用) | `docker ps -a` |
| Images | 本机所有镜像(名称 / tag / 大小 / 架构) | `docker images` |
| Volumes | 挂载的卷(名称 / 容量 / 用途) | `docker volume ls` |
| Builds | buildx 构建历史的只读查看(看耗时 / cache 命中 / 失败错误,不能从面板重跑 build) | `docker buildx history` |

</div>

&emsp;&emsp;本节先重点看 Containers / Images 两个模块,Volumes 会在第三章卷挂载时展开,Builds 跟着第四章 buildx 跨架构 build 时再回头看。

### 2.4 跑第一个 hello-world

&emsp;&emsp;`docker pull` 把镜像从仓库拉到本机,`docker run` 拿镜像起容器跑——这两条命令组成 Docker 最基础的 pull-then-run 闭环。我们用官方的 `hello-world` 镜像走一遍,这个镜像启动后会打印一段说明文字然后立即退出,正适合当入门样本。

```bash
docker pull hello-world
```

```bash
docker run hello-world
```

&emsp;&emsp;`docker pull` 把 `hello-world:latest` 这份镜像从 Docker Hub 拉到本机本地存储;`docker run` 把刚拉到的镜像启动成一个容器,执行镜像里预设的 CMD(/hello 二进制),打印一段问候文字之后容器立即退出。这两条命令演示了三对象闭环——仓库(`docker.io/library/hello-world`)→ 镜像(本机存储)→ 容器(运行实例)。任何 docker 上手第一步都是它,验证 daemon 能联通仓库、本机能拉镜像、能跑容器。

&emsp;&emsp;不同电脑不同系统运行 `docker run hello-world` 的输出很有可能是不同的，本机大概是下面这样:

```text
Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (arm64v8)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.
```

&emsp;&emsp;输出里第 2 步的 `(arm64v8)` 是关键——本机 Apple Silicon 机器拉到的是 arm64 架构的镜像。镜像本身带"架构"这个属性,本机 arm64 镜像不能直接拿去 x86 服务器跑起来。

> <font size=2>**【名词解释】<font color=red>arm64</font>(ARM 64-bit Architecture，ARM 64 位架构)** — CPU 指令集架构的一种，苹果 M 系列芯片（Apple Silicon）和绝大多数手机芯片使用这套指令集。Docker 镜像在 build 时会被编译成特定架构的二进制，arm64 镜像只能在 arm64 机器上跑。</font>

> <font size=2>**【名词解释】<font color=red>amd64</font>(AMD 64-bit Extension，又称 x86_64)** — Intel 和 AMD 服务器、桌面 CPU 使用的指令集架构。腾讯云、AWS 等云厂商的绝大多数 Linux 实例都是 amd64。本机是 arm64，腾讯云是 amd64——两边架构不同。</font>

### 2.5 命令行与界面对照

&emsp;&emsp;pull + run 跑完之后,我们切到 GUI 看一眼——Docker Desktop 左侧 Images tab 会立刻出现一行 `hello-world:latest 22.6 kB arm64`,Containers tab 会出现一行 `objective_lovelace Exited (0)`(容器名由 docker 随机生成,每人不一样)。

&emsp;&emsp;命令行那一侧同样可以看到:

```bash
docker images hello-world
```

&emsp;&emsp;Docker Desktop 4.73 / Engine 29.x 起，`docker images` 默认输出换成了新格式:

In [ ]:
IMAGE                ID             DISK USAGE   CONTENT SIZE   EXTRA
hello-world:latest   0e760fdfbc48       22.6kB         10.3kB    U

<style>.center{width:auto;display:table;margin-left:auto;margin-right:auto;}</style>
<p align="center"><font face="黑体" size=4>docker images 字段说明</font></p>
<div class="center">

| 字段 | 含义 |
|---|---|
| `IMAGE` | 镜像名 + tag，格式 `仓库名:tag` |
| `ID` | 镜像内容的 SHA256 哈希前 12 位；由镜像内容决定，不是随机生成——同一份镜像任何人 pull 下来 ID 都一样，可用来验证镜像完整性 |
| `DISK USAGE` | 本机磁盘实际占用（含各层共享后的净占用） |
| `CONTENT SIZE` | 镜像内容本身的大小（不含宿主机共享层） |
| `EXTRA` | `U` = In Use，有容器正在引用这份镜像；空 = 无容器引用，可以安全删除 |

</div>

&emsp;&emsp;老版本 Docker 输出的是 `REPOSITORY / TAG / IMAGE ID / CREATED / SIZE` 五列，数据含义一致，只是展示方式变了。

```bash
docker ps -a | head -3
```

<style>.center{width:auto;display:table;margin-left:auto;margin-right:auto;}</style>
<p align="center"><font face="黑体" size=4>docker ps 字段说明</font></p>
<div class="center">

| 字段 | 含义 |
|---|---|
| `CONTAINER ID` | 容器创建时随机生成的 64 位 SHA256 哈希，显示前 12 位；每次 `docker run` 都不同，即使用同一个镜像 |
| `IMAGE` | 这个容器是从哪个镜像启动的 |
| `COMMAND` | 容器启动时跑的命令（hello-world 跑的是 `/hello` 这个可执行文件） |
| `CREATED` | 容器创建时间 |
| `STATUS` | 容器当前状态；`Exited (0)` = 正常退出，括号里是退出码，0 表示成功；`Up X minutes` = 正在运行 |
| `PORTS` | 端口映射，格式 `宿主机端口 → 容器端口`；hello-world 不监听端口所以为空 |
| `NAMES` | 容器名，不指定时 docker 随机生成两个单词拼在一起（如 `objective_lovelace`） |

</div>

&emsp;&emsp;两条命令的输出和 GUI 表格的内容是逐字对应的——GUI 不是"另一个工具",它和命令行操作的是同一份对象,只是显示方式不同。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/2.5-%E5%91%BD%E4%BB%A4%E8%A1%8C%E4%B8%8EGUI%E5%AF%B9%E7%85%A7-be37de93.png" width=80%>
</div>

### 2.6 用 inspect 读取镜像元数据

&emsp;&emsp;`docker inspect` 把镜像或容器的完整元数据(JSON 格式)打出来——镜像 ID(sha256...)、架构、操作系统、创建日期、启动命令、镜像大小,一应俱全。我们用它看一眼 hello-world 的"身份证"。

```bash
docker inspect hello-world | head -30
```

> 注意：`docker pull` 刚完成后立刻跑 `inspect` 偶尔会报 `no such object`——daemon 的对象索引还没写完，重跑一次即可，不是真的出错了。

&emsp;&emsp;`inspect` 把镜像在本机存储里的元数据全部摊开——`head -30` 取前 30 行能看到这几个关键字段:`Id`(sha256... 镜像唯一标识)、`Architecture`(本机 Apple Silicon 显示 `arm64`)+ `Variant`(`v8`)、`Os`(`linux`)、`Cmd`(`["/hello"]` 启动命令)、`Size`(rootfs 内容字节数,hello-world 大约 17 kB)、`Created`(镜像构建时间戳)。镜像 = sha256 唯一标识的不可变对象,inspect 是后续撞"架构对不上"(第四章)、"分层缓存失效"(第三章)的第一手诊断工具。后面会看到 ai-todo(~72 MiB)、文搜图 backend(~5 GB)、frontend(~80 MB)各档体量,跟这十几 kB 形成小/中/大对照。

> 注意:GUI Images tab / `docker images` 显示的 22.6 kB 是 docker daemon 计算的<b>本机磁盘总占用</b>(含 manifest 元数据 + 内容),`inspect` 的 `Size` 字段(约 17 kB)只是 rootfs 内容大小,两者差几 kB 是元数据开销,都是真实数字、只是统计口径不同。

### 2.7 容器与镜像的清理

&emsp;&emsp;实操要养成"用完清掉"的习惯——容器留着不删,Docker Desktop 会越积越多容器残骸;镜像不删,磁盘也会被几十 GB 的旧镜像吃满。这里给一组完整清理命令,后续章节多次复用。

```bash
# 第一步:列出所有用 hello-world 跑过的容器 ID
docker ps -aq --filter ancestor=hello-world
```

```bash
# 第二步:把上一步列出的容器全部删掉
docker ps -aq --filter ancestor=hello-world | xargs -r docker rm
```

```bash
# 第三步:删镜像
docker rmi hello-world
```

```bash
# 第四步:精准验证 hello-world 已彻底清除(两条都期望空输出)
docker images hello-world
docker ps -a --filter ancestor=hello-world
```

&emsp;&emsp;四步完整清理——列容器、删容器、删镜像、精准验证。`docker rmi` 默认不允许删"还有容器在引用"的镜像,所以必须先 rm 容器再 rmi 镜像。第四步用两条按 hello-world 过滤的命令验证彻底清除——`docker images hello-world` 空输出代表镜像没了,`docker ps -a --filter ancestor=hello-world` 空输出代表容器也没了,这样跟本机其他镜像 / build cache 完全脱钩,数字不会被淹没。第三章 ai-todo 单容器实验、第五章文搜图 compose 实验都会反复用到这套清理流程,养成"用完即清"避免本机磁盘被旧镜像占满。

> 提示:想看本机整体资源占用(所有镜像 / 容器 / 卷 / build cache 的合计)可以加跑一条 `docker system df`——但本机如果还有其他镜像 / build cache,这条命令的数字主要由它们贡献,不是"验证 hello-world 释放成功"的对位工具。

&emsp;&emsp;**Windows PowerShell 完整四步**:`xargs` 在 PowerShell 不可用,改用数组变量 + 条件判断:

```powershell
# Windows PowerShell - 对照上面 mac/Linux 四步
# 第一步: 列容器(命令一致)
docker ps -aq --filter ancestor=hello-world

# 第二步: 删容器
$ids = docker ps -aq --filter ancestor=hello-world
if ($ids) { docker rm -f $ids }

# 第三步: 删镜像
docker rmi hello-world

# 第四步: 精准验证(两条都期望空输出)
docker images hello-world
docker ps -a --filter ancestor=hello-world
```

&emsp;&emsp;或者直接在 Git Bash 里跑跟 mac / Linux 一致的写法。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/2.7-%E5%AE%B9%E5%99%A8%E5%92%8C%E9%95%9C%E5%83%8F%E6%B8%85%E7%90%86%E9%A1%BA%E5%BA%8F-131d63fc.png" width=80%>
</div>

### 2.8 Resources 配置入口

&emsp;&emsp;点右上角齿轮图标打开 Settings，左侧栏选 <b>Resources（资源）</b>，进入 <b>Advanced（高级）</b> 标签页，能看到三个滑块：

- **CPU limit（CPU 上限）**：Docker 底层 Linux VM 最多能用几个核心。<b>默认是宿主机核心数的一半</b>（8 核机器默认给 4 核）。调到上限也只是上限，不会一直占满——容器空闲时核心照样归还给宿主机。<b>建议留 2 核给自身</b>，其余全给 Docker 问题不大。

- **Memory Limit（内存上限）**：分配给 Linux VM 的内存池，所有容器共用这块。<b>默认是宿主机内存的 50%</b>（16 GB 机器默认 8 GB）。跑 ai-todo 这类轻量单容器，4 GB 都够用；跑第五章文搜图（向量库 + MinIO + 多个服务同时起），<b>建议调到 12 GB 以上</b>，否则容器容易被 OOM 杀掉。

- **Swap（交换空间）**：内存真的不够用时拿硬盘顶上，速度慢但能防止容器直接崩。<b>默认 1 GB，保持不动即可</b>——真正缺内存还是应该调上面的 Memory Limit，而不是靠 Swap 救急。

&emsp;&emsp;跑完本课前几章不需要动这些数值。到第五章文搜图之前，记得把 Memory 调到 12 GB 以上再启动 compose。

### 2.9 第二章命令速查

&emsp;&emsp;本章涉及的命令集中在这里，后续章节随时翻查。

&emsp;&emsp;**镜像操作**

<style>.center{width:auto;display:table;margin-left:auto;margin-right:auto;}</style>
<p align="center"><font face="黑体" size=4>镜像操作命令汇总</font></p>
<div class="center">

| 命令 | 作用 |
|---|---|
| `docker pull <镜像名>` | 从 Registry 拉取镜像到本机 |
| `docker images` | 列出本机所有镜像 |
| `docker images <镜像名>` | 只列出指定镜像 |
| `docker inspect <镜像名>` | 查看镜像完整元数据（JSON） |
| `docker rmi <镜像名>` | 删除镜像（有容器引用时会报错，先删容器） |
| `docker image prune -f` | 删所有没有 tag 的悬空镜像 |

</div>

&emsp;&emsp;**容器操作**

<style>.center{width:auto;display:table;margin-left:auto;margin-right:auto;}</style>
<p align="center"><font face="黑体" size=4>容器操作命令汇总</font></p>
<div class="center">

| 命令 | 作用 |
|---|---|
| `docker run <镜像名>` | 从镜像创建并启动容器 |
| `docker run -d <镜像名>` | 后台运行 |
| `docker run --name <名字> <镜像名>` | 指定容器名（不指定则随机生成） |
| `docker ps` | 列出正在运行的容器 |
| `docker ps -a` | 列出所有容器（含已停止的） |
| `docker stop <容器名/ID>` | 优雅停止容器（发 SIGTERM，等 10s 后 SIGKILL） |
| `docker rm <容器名/ID>` | 删除已停止的容器 |
| `docker rm -f <容器名/ID>` | 强制停止并删除（等价于 stop + rm） |
| `docker logs <容器名/ID>` | 查看容器日志 |
| `docker logs -f <容器名/ID>` | 实时跟踪日志 |
| `docker exec -it <容器名/ID> bash` | 进入容器内部开一个交互式 shell |
| `docker inspect <容器名/ID>` | 查看容器完整元数据（JSON） |

</div>

&emsp;&emsp;**空间管理**

<style>.center{width:auto;display:table;margin-left:auto;margin-right:auto;}</style>
<p align="center"><font face="黑体" size=4>空间管理 / 清理命令汇总</font></p>
<div class="center">

| 命令 | 作用 |
|---|---|
| `docker system df` | 查看镜像/容器/卷占用的磁盘空间 |
| `docker system prune -f` | 删所有停止的容器 + 悬空镜像 + 无用网络 |
| `docker system prune -a -f` | 在上面基础上额外删所有没有容器引用的镜像 |

</div>

---

## <center>第三章 ai-todo 单容器跑通</center>

&emsp;&emsp;第二章在 hello-world 这个 22.6 kB 的官方镜像上跑通了 pull-run-清理的最小闭环。本章带前置课已经熟悉的 ai-todo 进容器——写一份 Dockerfile,把"venv 装依赖 + systemd 起进程"换成"docker build 出镜像 + docker run 起容器",并通过前端 UI 完成持久化验证。

&emsp;&emsp;<b>实操前置:先在本机用 venv 把 ai-todo 跑通</b>。容器化之前我们先确认这个项目本机能跑、API key 配置正确、前端能访问后端——这样后面进容器撞错时,我们才能区分"是 ai-todo 自己的问题"还是"容器化引入的新问题"。三步:建 venv + 装依赖、配 `.env`、双终端起后端和前端。

```bash
# === macOS / Linux / Git Bash:终端 A 起后端(项目根目录) ===
cd ai-todo
python3.12 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple

# 配 .env:填上自己的 OPENROUTER_API_KEY(申请: https://openrouter.ai/keys)
cp .env.example .env
# 编辑 .env,把 OPENROUTER_API_KEY 那行换成真实 key 后保存

# 启动后端
uvicorn main:app --reload --port 12234
```

```powershell
# === Windows PowerShell:终端 A 起后端 ===
cd ai-todo
py -3.12 -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple

copy .env.example .env
# 用 notepad / VS Code 打开 .env,填 OPENROUTER_API_KEY

uvicorn main:app --reload --port 12234
```

```bash
# === 三平台共用:终端 B 另开,起前端静态服务器 ===
cd ai-todo
python3.12 -m http.server 12233 --directory frontend
# Windows 用 py -3.12 -m http.server 12233 --directory frontend
```

&emsp;&emsp;后端启动看到 `[lifespan] agent 就绪` + `Uvicorn running on http://0.0.0.0:12234`,前端启动看到 `Serving HTTP on :: port 12233`,两个都到位了。浏览器开 `http://localhost:12233`,左侧能新建会话,中间发一句"列出所有 todos"或"加一个明天写周报",右侧月度日历能看到对应日期标记——这就说明 ai-todo 本机跑通了,可以进入容器化主线。跑通之后两个终端都 `Ctrl+C` 停掉,我们下面要把这套手敲流程交给 Dockerfile 自动完成。

> 提示:PowerShell 首次激活 venv 报"无法加载...因为在此系统上禁止运行脚本"时,跑一次 `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned` 后再试。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/3.0-ai-todo%E7%95%8C%E9%9D%A2-b104f06f.png" width=80%>
</div>

### 3.1 Dockerfile 与手动命令对照

&emsp;&emsp;ai-todo 目前没有 Dockerfile——这是一个纯文本文件，类似一个特殊的 `.sh` 脚本：把"装 Python、装依赖、放源码、起进程"这些步骤写下来，`docker build` 一次自动跑完，产出一个可分发的镜像。在裸机上手敲这些命令的做法，换台机器就得重来一遍，还得靠文档和记忆；Dockerfile 把步骤锁进文件，任何人拿到都能跑出完全一样的环境。本章第一件事就是为 ai-todo 创建这份文件。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/3.1-Dockerfile%E6%98%AF%E4%BB%80%E4%B9%88-e68c1ad1.png" width=80%>
</div>

&emsp;&emsp;动手之前，先把手动部署的操作和 Dockerfile 指令对照一遍：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>手动操作 vs Dockerfile 指令对照</font></p>
<div class="center">

| 手动操作 | 等效的 Dockerfile 指令 |
|---|---|
| 装 Python 3.12 | `FROM python:3.12-slim` |
| `cd /srv/ai-todo` | `WORKDIR /app` |
| 建非 root 用户跑服务 | `RUN useradd app` + `USER app` |
| `python -m venv .venv` 建虚拟环境 | (镜像本身就是隔离环境,不需要再建 venv) |
| `pip install -r requirements.txt` 装依赖 | `COPY requirements.txt .` + `RUN pip install -r requirements.txt` |
| `scp` / `git pull` 上传源码 | `COPY . /app` |
| 手敲一行 `uvicorn main:app --host 0.0.0.0 --port 12234` 起服务 | `CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "12234"]` |
| 端口暴露(裸机进程直接监听宿主机端口,不需要声明) | `EXPOSE 12234`(容器内监听端口,声明给读者看 + `docker run -p` 映射依据) |

</div>

&emsp;&emsp;对照下来,Dockerfile 干的事情和手动敲命令是一样的——装解释器、装依赖、放源码、起进程。区别在于:手动敲是每台机器都要重来一遍、还得靠文档和记忆;Dockerfile 把这些步骤写成文本文件,`docker build` 一次产出可分发的镜像,换机器只要拉镜像就行。

### 3.2 Dockerfile 关键指令速览

&emsp;&emsp;动手写第一份 Dockerfile 之前,先把本课会用到的核心指令过一遍。Dockerfile 是一份"按行从上到下执行"的脚本,每一行是一条指令,`docker build` 时每条指令产生一层 layer——这种分层结构让缓存复用变得自然。下面这张表是后面 3.3 节看 ai-todo 实际怎么组装这些指令前的语法字典。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Dockerfile 核心指令速查</font></p>
<div class="center">

| 指令 | 干啥 | 典型用法 |
|---|---|---|
| `FROM` | 选基础镜像,后续所有指令叠在这层上 | `FROM python:3.12-slim` |
| `WORKDIR` | 设容器内工作目录,后续 `COPY` / `RUN` / `CMD` 都以此为根 | `WORKDIR /app` |
| `COPY` | 把构建上下文(本机当前目录)的文件拷进镜像 | `COPY requirements.txt .` |
| `ADD` | 类似 COPY 但额外支持自动解压 tar / 拉远程 URL | `ADD https://... /tmp/`(用得少,推荐 COPY) |
| `RUN` | build 时执行一条命令,改动会进镜像层 | `RUN pip install -r requirements.txt` |
| `ENV` | 设环境变量(build 期和 runtime 都生效) | `ENV PYTHONUNBUFFERED=1` |
| `USER` | 切换后续指令的执行用户(默认 root) | `USER app` |
| `EXPOSE` | 声明容器监听端口(只是文档,不实际开端口) | `EXPOSE 12234` |
| `HEALTHCHECK` | 容器内周期跑探针,根据退出码标 healthy / unhealthy | `HEALTHCHECK CMD curl -f http://localhost:12234/api/health` |
| `CMD` | 容器启动时跑的命令(每镜像一条,可被 `docker run` 覆盖) | `CMD ["uvicorn", "main:app", "--host", "0.0.0.0"]` |
| `ENTRYPOINT` | 类似 CMD,但不能被 `docker run` 完全覆盖,常用来固定入口 | `ENTRYPOINT ["uvicorn"]`(本课用 CMD 不用 ENTRYPOINT) |

</div>

> 提示:这些指令每条会产生一层 layer,改动越靠后的层、build 重新执行的范围越小。所以最佳实践是 `FROM` / `WORKDIR` / `COPY requirements + RUN pip install` 这种"基础 + 依赖"的层放前面,`COPY 源码` 这种经常变的层放最后。

### 3.3 第一版 Dockerfile

&emsp;&emsp;接续 3.2 的指令字典,我们看 ai-todo 怎么把这些指令组装出一份完整的 Dockerfile。`docker build` 从上到下依次执行,每一条指令各司其职。我们在 ai-todo 项目根目录下新建一个名为 `Dockerfile` 的文件(没有后缀名):

```bash
cd ai-todo项目目录
touch Dockerfile
```

&emsp;&emsp;把上面这些指令组合起来，填进去就是 ai-todo 的第一版 Dockerfile：

```dockerfile
# 基础镜像:Python 3.12 的 slim 版本,带 pip 但不含 build-essential,体积约 80 MB
FROM python:3.12-slim

# Python 运行时环境变量
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1                    # 关 .pyc 字节码生成 + 关 stdout 缓冲让日志实时显示

# 设置容器内工作目录,后续 COPY / RUN / CMD 都以这里为基准(等效 mkdir -p + cd)
WORKDIR /app

# 先单独拷 requirements.txt —— 依赖没变时这一层缓存可复用,重 build 快很多
COPY requirements.txt .                   # 只拷依赖清单,不拷源码,做 layer cache 优化

# 装 Python 依赖,走清华 mirror 加速国内拉包(pypi.org 国内慢)
RUN pip install -i https://pypi.tuna.tsinghua.edu.cn/simple -r requirements.txt

# 再拷源码(.dockerignore 已经把 .env / .venv / data / __pycache__ 等排除)
COPY . /app                               # 源码改一行只会让本层重建,上面装依赖层命中缓存

# 建普通用户跑服务,不用 root(生产安全最佳实践)
RUN groupadd --system --gid 1000 app && \
    useradd --system --uid 1000 --gid app --home-dir /app --shell /usr/sbin/nologin app && \
    mkdir -p /app/data /app/logs && \     # 建数据目录(挂卷用)和日志目录
    chown -R app:app /app                 # 整个 /app 归 app 用户所有,容器内 app 才能写

# 后续指令(CMD)用 app 用户身份执行,而不是 root
USER app

# 声明容器会监听的端口(只是文档标注,docker run -p 才真正映射)
EXPOSE 12234

# 容器启动命令(exec 形式):uvicorn 直接作为 PID 1,docker stop 的 SIGTERM 信号能直达
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "12234"]
```

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/3.3-%E7%AC%AC%E4%B8%80%E7%89%88dockerfile-0da534ab.png" width=80%>
</div>

&emsp;&emsp;几个值得注意的细节：

- **`PYTHONUNBUFFERED=1`**——关掉 Python 的输出缓冲，容器日志立刻写到标准输出，`docker logs` 才能看到实时内容。

- **先 COPY requirements.txt 再 COPY 源码**——Docker 每一条 `RUN`/`COPY` 指令都会生成一个缓存层。requirements.txt 没变时，pip install 那一层直接复用缓存，改了业务代码重 build 只需要几秒。

- **`USER app`**——容器内进程以普通用户 UID 1000 跑，而不是 root，安全边界更清晰。

- **`CMD ["uvicorn", ...]` 数组形式**——uvicorn 直接作为容器 PID 1，`docker stop` 发的停止信号能直达它，不会因为套了一层 shell 而收不到。

### 3.4 .dockerignore 排除清单

&emsp;&emsp;Dockerfile 里有一行 `COPY . /app`,把项目当前目录全部内容拷进镜像。但项目目录里有些东西**不应该进镜像**——比如 `.env` 文件(含 API key 这种机密)、`.venv*` 目录(本机的虚拟环境,容器内会重新装)、`data/` 和 `logs/`(运行时目录,容器内会重建)、`__pycache__/`(Python 字节码缓存,镜像里浪费空间)。

&emsp;&emsp;所以还需要在 ai-todo 项目根目录下新建一份 `.dockerignore`——类似 `.gitignore` 的语法，告诉 docker build 哪些路径不要打进 build context：

```bash
touch .dockerignore
```

&emsp;&emsp;内容如下:

```text
# 配置 / 机密 - 通过 --env-file 注入, 严禁打包进镜像（镜像推到 Hub 任何人都能看到）
.env
.env.*
!.env.example

# Python 虚拟环境 / 缓存
.venv
.venv-*
venv
__pycache__
*.py[cod]
*$py.class
.pytest_cache
*.egg-info

# 运行时数据 / 日志 - 通过卷挂载, 不进镜像
data
logs

# Git / IDE
.git
.gitignore
.vscode
.idea
*.swp
.DS_Store

# Docker 自身产物
Dockerfile
.dockerignore
docker-compose*.yml

# 文档 / 杂项
*.md
!SETUP.md
```

&emsp;&emsp;`.dockerignore` 阻止上述路径进入 docker build 的上下文(build context),`COPY . /app` 在内部会自动跳过这些路径。不让机密(`.env`)进镜像是安全的第一道防线——镜像 push 上 Docker Hub 后任何人能 pull,一旦 `.env` 进了镜像,API key 就泄露了。`.env` 必须在容器启动时通过 `--env-file` 注入,而不是打包进镜像,这是本章最关键的踩坑点,3.6 节的 `docker run` 主线会展开。

### 3.5 build 看 layer 缓存

&emsp;&emsp;Docker 在构建镜像时,会把<b>会改变文件系统的步骤</b>保存成一层层 layer。比如 `COPY` 会把文件拷进镜像、`RUN pip install` 会安装依赖,这些都会形成新的文件系统差异层。每一层只记录相对于上一层新增、修改或删除了哪些内容。最终镜像就是<b>基础镜像层 + 我们自己构建出来的层叠加后的结果</b>。

> 注意:`ENV` / `WORKDIR` / `USER` / `EXPOSE` / `CMD` 这类指令更多是<b>镜像配置</b>,不要简单理解成文件系统层。

&emsp;&emsp;但是,<b>层和缓存不是一回事</b>。Dockerfile 是从上到下构建的,只要前面某一步的内容或构建状态发生变化,从这一步往后的缓存通常都会失效。后面的步骤即使自己没有改,也要基于新的前置状态重新计算:如果是 `COPY` / `RUN`,就可能重新生成文件系统层;如果是 `USER` / `EXPOSE` / `CMD`,也会重新写入新的镜像配置。

&emsp;&emsp;这就是为什么 Dockerfile 里 `COPY requirements.txt .` 要放在 `COPY . /app` 前面——requirements.txt 不动时,后面的 `RUN pip install` 层就能命中缓存被跳过;只有业务代码那层重新执行,build 时间从分钟级降到秒级。如果反过来先 `COPY . /app`,改一行业务代码就会让 pip install 那层缓存失效、重新跑。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/3.5-%E9%95%9C%E5%83%8F%E5%88%86%E5%B1%82%E4%B8%8Elayer%E7%BC%93%E5%AD%98-6d1f6b96.png" width=80%>
</div>

&emsp;&emsp;Dockerfile 写好了,我们用 `docker build` 把它构建成镜像。第一次 build 会拉 base 镜像(`python:3.12-slim`)、装所有 pip 依赖,大概需要 1-3 分钟;第二次 build(假设源码改了但 requirements.txt 没动),layer cache 会让 pip install 这一步直接复用缓存,build 时间降到几秒钟。

```bash
# 进 ai-todo 项目根目录 (替换成你本机的真实路径)
cd /path/to/ai-todo

# 第一次 build
docker build -t ai-todo:0.1 .
```

&emsp;&emsp;把当前目录(`.`)作为 build context 提交给 docker daemon,按 Dockerfile 一步步执行,产出 tag 为 `ai-todo:0.1` 的镜像。`-t ai-todo:0.1` 指定镜像名和 tag(tag 默认是 `latest`,显式写出版本号便于后续管理)。<b>第一次 build</b> 需要拉取 `python:3.12-slim` 基础镜像、跑完 pip install 所有依赖，耗时 1-3 分钟；<b>之后只要 `requirements.txt` 没变</b>，pip install 那层命中缓存直接跳过，build 从分钟级降到几秒——上面输出里 `CACHED [3/6]` `CACHED [4/6]` 就是缓存命中的标志。

&emsp;&emsp;build 完成后看一下镜像列表:

```bash
docker images ai-todo
```

&emsp;&emsp;build 完成后用 `docker images ai-todo` 查看镜像大小。实测 ai-todo:0.1 在本机 arm64 build 后 `docker images` 显示约 71.8 MiB(amd64 跨架构 build 出来约 72.3 MiB,两者差异主要在 CPU 指令集层的二进制);如果用 `docker system df` 看本机总占用空间会更大,因为 Docker Desktop 会按"含基础镜像共享层"汇总。ai-todo 带了 langchain / langgraph / openai 等依赖,pip install 层就是镜像体积的主要来源。

### 3.6 docker run 启动容器

&emsp;&emsp;镜像 build 出来了，用 `docker run` 把它跑起来。`.dockerignore` 已经把 `.env` 排在镜像外，所以启动时必须通过 `--env-file` 把配置从宿主机注入进去——这是容器化的标准做法：**镜像只装代码和依赖，环境配置在运行时注入**，同一份镜像可以对接不同环境的 `.env`，也不用担心 API key 随镜像泄露。

```bash
# macOS / Linux / Git Bash
docker run -d --name ai-todo \
    -p 12234:12234 \
    --env-file .env \
    ai-todo:0.1
```

```powershell
# Windows PowerShell - 续行符用反引号 ` 替代反斜杠 \
docker run -d --name ai-todo `
    -p 12234:12234 `
    --env-file .\.env `
    ai-todo:0.1
```

&emsp;&emsp;几个参数说明：`-d` 后台运行；`--name ai-todo` 给容器起固定名字便于后续操作；`-p 12234:12234` 把容器内 12234 端口映射到宿主机 12234，前置课的 nginx 反代配置（`proxy_pass http://127.0.0.1:12234`）一行不用改；`--env-file` 把指定文件里所有 `KEY=value` 行注入容器的环境变量表。

&emsp;&emsp;启动后用 `docker logs` 确认服务正常起来了：

```bash
docker logs ai-todo
```

&emsp;&emsp;看到下面这几行说明启动成功：

```text
INFO:     Started server process [1]
INFO:     Waiting for application startup.
HH:MM:SS.xxx [INFO] [lifespan] 加载配置 — model=z-ai/glm-5.1  port=12234
HH:MM:SS.xxx [INFO] [lifespan] agent 就绪
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:12234 (Press CTRL+C to quit)
```

### 3.7 卷挂载持久化

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/3.7-%E5%8D%B7%E6%8C%82%E8%BD%BD%E6%8C%81%E4%B9%85%E5%8C%96-35072243.png" width=80%>
</div>

> <font size=2>**【名词解释】<font color=red>卷（Volume，存储卷）</font>** — Docker 官方文档定义是"独立于容器生命周期的持久化数据存储机制"。容器本身的文件系统是一次性的(容器删了里面写的文件就没了),卷把"需要保留的数据"放到容器之外的位置——可以是宿主机磁盘上的目录(叫 bind mount,直接绑定),也可以是 Docker 管理的命名卷(named volume,Docker 自己分配存储位置)。本节我们用的就是 bind mount:`-v 宿主机路径:容器内路径`,容器进程往容器内路径里写,文件实际落在宿主机磁盘上。</font>

&emsp;&emsp;ai-todo 的数据(todos)存在 SQLite 数据库文件 `/app/data/todos.db` 里——这是容器内路径。如果不挂卷,容器删掉之后这份数据就一起没了。我们用 `-v` 把宿主机的目录挂载到容器内的 `/app/data`,这样数据库文件实际写到宿主机磁盘,容器删了重新启动还能读到上一次的数据。

```bash
# 停掉 3.6 起的容器，加上卷挂载重新启动
docker rm -f ai-todo

mkdir -p ~/docker-data/ai-todo

docker run -d --name ai-todo \
    -p 12234:12234 \
    --env-file .env \
    -v ~/docker-data/ai-todo:/app/data \
    ai-todo:0.1
```

&emsp;&emsp;`-v 宿主机绝对路径:容器内路径` 把宿主机 `/Users/xiaoyang/docker-data/ai-todo` 这个目录挂载到容器内的 `/app/data`——容器进程在 `/app/data` 下读写文件,实际操作的是宿主机磁盘上的对应路径。这种挂载叫"bind mount"——直接绑定宿主机路径,容器看到的内容和宿主机一致。数据(todos.db)留在宿主机,容器可以随时删 / 升级 / 重启,数据永远在;后续第四章 4.8 节把 ai-todo 在腾讯云上拉起时,只需要把这个数据目录用 scp 同步过去,容器在云上启动后能继续接读本机最后一次的状态。

&emsp;&emsp;**Windows PowerShell 完整写法**:`mkdir -p` 改 `New-Item`,续行符 `\` 改反引号 `` ` ``,路径用 `$env:USERPROFILE`:

```powershell
# Windows PowerShell - 卷挂载完整写法
docker rm -f ai-todo
New-Item -ItemType Directory -Force -Path "$env:USERPROFILE\docker-data\ai-todo"

docker run -d --name ai-todo `
    -p 12234:12234 `
    --env-file .\.env `
    -v "$env:USERPROFILE\docker-data\ai-todo:/app/data" `
    ai-todo:0.1
```

冒号分隔的前半是 Windows 宿主机路径、后半保持容器内的 Unix 路径,Docker Desktop 会自动转换。

> <font size=2><b>【提示】挂载目录的权限,本机不撞 / Linux 服务器一定撞</b> — 我们的 ai-todo 镜像里跑的是 `app` 用户(Dockerfile 里 `USER app`),不是 root。本机 Docker Desktop 这一节不会出问题——Mac/Windows 上 docker 通过 osxfs/gvisor 文件系统层做了权限映射,容器内非 root 用户写宿主机目录是放行的。但是到了第四章云端 Linux 服务器 + 第五章文搜图本机跑(因为是 compose 起多容器、user 严格)就会撞:docker 自动创建的宿主机挂载目录归 root 所有,容器内的 app 用户<b>没有写权限</b>,SQLite 在 `/app/data/` 下建 `todos.db` 时报 `unable to open database file`,容器立刻 lifespan 失败退出。<br><br><b>通用修法两步</b>:① 先查容器内 app 用户的真实 UID/GID(`useradd --system` 给出的具体数字跟 base 镜像 + 内核版本相关,可能是 999 也可能是 1000);② 把宿主机目录所有权改成那个数字。<br><br>`docker run --rm <镜像名> id app` ← 查 app 用户的 uid/gid<br>`sudo mkdir -p <宿主机挂载目录> && sudo chown -R <uid>:<gid> <宿主机挂载目录>`<br><br>本课实测 ai-todo 镜像 app 用户是 `uid=1000(app) gid=1000(app)`,所以云端的 chown 用 `1000:1000`。这条 chown 在第四章 4.8 节 ai-todo 上腾讯云时还会再次出现——见到容器反复 restart + 日志报 `Permission denied` 或 `unable to open database file`,第一反应就是查这个。</font>

### 3.8 前端 UI 持久化验证

&emsp;&emsp;数据持久化挂好了,我们验证一下。验证的关键不是"容器重启数据还在"——`docker restart` 只是把同一个容器停了再启,容器本身没删,数据当然在。真正要证明的是<b>"容器删掉重建,数据还在"</b>——只有数据写到宿主机磁盘上,删容器才不会丢。所以验证步骤是:<b>前端添加一条 todo → `docker rm -f` 删容器 → 用同样的 `-v` 重新 `docker run` → 浏览器刷新,看 todo 还在不在</b>。

&emsp;&emsp;先把前端跑起来。ai-todo 项目目录下有 `frontend/` 单页应用,前置课用 nginx 反代把前端挂在 :80,容器化阶段我们临时用 Python 自带的静态服务器跑前端,这样不需要再起一个 nginx 容器:

```bash
# 在 ai-todo 项目根目录另开一个终端,起静态服务跑前端
cd ai-todo
python3 -m http.server 12233 --directory frontend
```

&emsp;&emsp;`python3 -m http.server` 是 Python 标准库自带的 HTTP 静态文件服务器,`--directory frontend` 指定服务 `frontend/` 这个目录的内容,绑定到本机 12233 端口。前端 `app.js` 里的 `BACKEND` 配置写好了——前端跑在 12233 端口时自动把 API 请求指向 12234(容器映射的后端端口),浏览器打开 **http://localhost:12233** 就能看到 ai-todo 界面。

&emsp;&emsp;**第一步:前端添加 todo**。在 ai-todo 界面的 chat 框输入"明天上午 10 点提醒我开周会",回车。智能体会调 `add_todo` 工具往 SQLite 写一条记录,右侧 todo 列表里立刻出现这条 todo。这是因为前端把消息发到容器内 `:12234/chat/stream`,LangGraph agent 根据自然语言决定调用 `add_todo` 工具,工具直接写 `/app/data/todos.db`——这个路径在容器内,但实际通过 `-v` 挂载落到了宿主机 `~/docker-data/ai-todo/todos.db`。

&emsp;&emsp;**第二步:删容器**。

```bash
# -f 表示先 SIGKILL 再删除,不管容器当前在不在跑
docker rm -f ai-todo
```

```bash
# 验证容器真的没了
docker ps -a --filter name=ai-todo
```

&emsp;&emsp;`docker ps -a` 列出所有容器(含停止的),`--filter name=ai-todo` 只看名字含 ai-todo 的——输出应该是空的,说明容器彻底删除。

&emsp;&emsp;**第三步:用同样的 `-v` 重新 `docker run`**。

```bash
docker run -d --name ai-todo \
    -p 12234:12234 \
    --env-file .env \
    -v ~/docker-data/ai-todo:/app/data \
    ai-todo:0.1
```

&emsp;&emsp;这是一个**全新的容器实例**(`docker ps` 看会发现 CONTAINER ID 跟之前完全不一样),用同样的镜像、同样的端口映射、同样的 env 文件、同样的卷挂载。新容器内 `/app/data` 仍然挂载到宿主机 `~/docker-data/ai-todo`——也就是上一个容器写过 todo 的那个目录。

&emsp;&emsp;<b>第四步:浏览器刷新前端</b>。打开 http://localhost:12233 (如果还开着就刷新页面)。新建一个 session,看 todo 列表——刚才那条"明天上午 10 点开周会"应该完整出现在列表里。<b>这就证明了 `-v` 卷挂载的持久化能力——旧容器删了、新容器重建了、但数据没丢</b>,因为数据实际写在宿主机 `~/docker-data/ai-todo/todos.db`,跟容器生命周期解耦了。

&emsp;&emsp;<b>为什么持久化是刚需,不是可选</b>——刚才"删容器 + 重建"看起来像是刻意演练,实际是项目日常会反复遇到的场景。任何项目都会迭代:改了几行代码、升级了依赖、修了一个 bug,都要重新 `docker build` 出新镜像、然后停掉旧容器、用新镜像 `docker run` 起新容器。每次这套操作里,<b>旧容器都会被销毁,如果数据存在容器可写层里,跟着旧容器一起没了</b>——用户辛辛苦苦记下的 todo、聊天历史、上传的文件全清空,这是任何线上服务都接受不了的事故。卷挂载把数据放在宿主机磁盘上,容器只是"代码 + 运行时"的载体——代码可以随时换(新镜像),数据始终在原地。这就是容器圈说的 stateless 容器 + stateful 数据,本课的 ai-todo、第五章的文搜图、第六章的云端部署都遵循这套模式。

&emsp;&emsp;**程序化验证(可选)**——如果不想开浏览器,在容器内直接查 SQLite 数据库也行:

```bash
# ai-todo 镜像基于 python:3.12-slim,自带 Python 标准库 sqlite3 模块
# 任何 python 基镜像都直接可用,不需要 apt 装额外工具
docker exec -i ai-todo python <<'PY'
import sqlite3
con = sqlite3.connect('/app/data/todos.db')
rows = list(con.execute('SELECT id, title, due_date, due_time, done FROM todos'))
for r in rows:
    print(r)
print(f'共 {len(rows)} 条记录')
PY
```

&emsp;&emsp;**Windows PowerShell 写法**:heredoc `<<'PY'` 是 Unix shell 语法,PowerShell 不识别。等价用 here-string `@"..."@`:

```powershell
# Windows PowerShell - here-string 等价 heredoc
docker exec -i ai-todo python -c @"
import sqlite3
con = sqlite3.connect('/app/data/todos.db')
rows = list(con.execute('SELECT id, title, due_date, due_time, done FROM todos'))
for r in rows:
    print(r)
print(f'共 {len(rows)} 条记录')
"@
```

PowerShell here-string 以 `@"` 单独一行开头、`"@` 单独一行结尾。Git Bash 也可以直接跑上面 Unix 版本,跟 macOS / Linux 一致。

&emsp;&emsp;ai-todo 在本机的容器里跑通了:镜像 build 出来、容器起来、数据通过卷挂载跨容器重建仍然存活。但这个镜像目前只在本机,云端 amd64 服务器还不知道这份镜像的存在。下一章做两件事:把镜像 build 成云端能跑的 amd64 架构、推到 Docker Hub,再从云端 pull 下来跑起来。

---

## <center>第四章 镜像走出本机:跨架构 build + Docker Hub + 腾讯云拉起</center>

&emsp;&emsp;第三章里我们把 ai-todo 在本机容器里跑通了,但有一个隐含前提没说破——本机是 Apple Silicon(arm64 架构),build 出来的镜像也是 arm64 架构的。腾讯云上那台准备跑 ai-todo 的服务器是 amd64 架构的,这份 arm64 镜像直接 push 上去再拉下来跑会出问题。本章把单容器线的"分发"环节收口:在本机 build 出腾讯云能跑的 amd64 镜像、推到 Docker Hub、腾讯云 pull 下来跑起来,而且前置课的 nginx 配置一行不改地接上来。

&emsp;&emsp;路径是这样的——先讲架构差异(arm64 vs amd64)→ 撞架构错的后果 → buildx 跨架构 build → `--load` 和 `--push` 的分工 → docker login + push 到 Docker Hub → Docker Hub 的限额规则 → 腾讯云内网 mirror 加速 → 腾讯云 docker run 拉起 → 前置课 nginx 一行不改。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/4.0-%E8%B7%A8%E6%9E%B6%E6%9E%84%E6%9E%84%E5%BB%BA%E5%88%86%E5%8F%91%E6%B5%81%E7%A8%8B-c360ebe0.png" width=80%>
</div>

### 4.1 arm64 vs amd64 差异

&emsp;&emsp;先把"架构"这个词说清楚。<b>架构</b>这里特指 CPU 指令集架构(ISA, Instruction Set Architecture)——CPU 不是直接执行机器码字符串,而是按一套指令集解码二进制(`ADD` / `MOV` / `JMP` 这些机器指令的二进制编码格式)。不同厂商定义的指令集格式不同,编译出来的二进制文件不能跨架构跑。两套主流架构对照如下:

> <font size=2>**【名词解释】<font color=red>ISA(Instruction Set Architecture,指令集架构)</font>** — CPU 厂商定义的"机器码格式 + 寄存器布局 + 内存访问规则"标准。同一份 C/C++/Rust 源码,编译到 amd64 ISA 和 arm64 ISA 会得到两份完全不同的机器码,互相不能识别。</font>

<style>.center{width:auto;display:table;margin-left:auto;margin-right:auto;}</style>
<p align="center"><font face="黑体" size=4>两套主流 CPU 架构对照</font></p>
<div class="center">

| 架构名 | 别名 | 代表 CPU | 谁在用 |
|---|---|---|---|
| **amd64** | x86_64 | Intel Core / Xeon、AMD Ryzen | 绝大多数 PC、Linux 服务器、云主机 |
| **arm64** | aarch64 | Apple M1/M2/M3/M4、AWS Graviton、树莓派 | 苹果电脑、ARM 服务器、移动设备 |

</div>

&emsp;&emsp;两个架构的关系类似"中文和英文"——同样表达"加法"这件事,但二进制编码格式不同,amd64 编出来的可执行文件,arm64 CPU 看不懂。Docker 镜像里那些二进制——Python 解释器、torch 这种 C 扩展、PyPI 依赖里的 `.so` 文件——都是按 build 时的 CPU 架构编译出来的,**只能在同架构 CPU 上跑**。这是 ISA 层面的物理约束,不是 Docker 自己能绕开的。

&emsp;&emsp;本课对应的具体情况是:<b>本机 Apple Silicon Mac = arm64</b>,<b>腾讯云默认 CVM = amd64</b>,两台机器架构不一致。本机 `docker build` 出来的镜像默认带本机架构(arm64),直接 push 到 Hub 再到腾讯云 pull 下来跑会撞架构错——下一节看这种错配的后果。Linux / Windows 在 x86_64 PC 上跑的同学走的是"本机 amd64 build → 云端 amd64 跑"的同架构路径,<b>不会撞这个坑</b>;但只要你用的是 Apple Silicon、ARM 笔记本(部分 Windows on ARM),就必须处理跨架构 build,这是本章后半段 buildx 的主要存在意义。

&emsp;&emsp;先在本机看一眼镜像里的架构信息:

```bash
docker inspect ai-todo:0.1 | grep -E '"Architecture"|"Os"'
```

&emsp;&emsp;`docker inspect` 输出镜像的完整 manifest,我们 grep 出其中 `Architecture` 和 `Os` 两个字段。这是验证镜像架构归属的标准命令,push 前看一眼,push 错了能立刻发现。本机 build 的 ai-todo:0.1 会返回 `"Architecture": "arm64"`,如果按这份镜像直接 push 到 Hub 再在腾讯云 amd64 主机上 pull 下来跑,大概率不工作——下一节看这种错配的后果。

### 4.2 架构错配:在腾讯云跑不起来

&emsp;&emsp;arm64 镜像 push 到 Hub、再在腾讯云 amd64 主机上 `docker run`,容器进程根本起不来——裸 Linux 没有仿真层兜底,kernel 看到一份不认识架构的二进制直接拒绝执行,报 `exec format error` 立崩。本机 Apple Silicon 上跑 amd64 镜像看起来"能跑"是个错觉,那是 Docker Desktop 默认开了 QEMU 透明仿真兜底,会打一行 `WARNING: ... platform (linux/amd64) does not match host (linux/arm64/v8)` 然后照样把进程拉起来,仿真也比原生慢 5-10 倍。<b>本机的 QEMU 仿真不会跟我们上云</b>,本机能跑 ≠ 腾讯云能跑,所以镜像必须主动 build 成 amd64,下一节我们就用 buildx 解决。

> <font size=2>**【名词解释】<font color=red>QEMU 仿真</font>**(QEMU,Quick EMUlator,通用开源机器仿真器)— QEMU 官网定义是"通用的开源机器仿真器与虚拟机"。Docker Desktop 出厂自带 QEMU 透明仿真层,让 arm64 主机能 `docker run` amd64 镜像(反之亦然),代价是仿真比原生慢 5-10 倍。这层只在 Docker Desktop 这种"为开发体验设计"的产品里默认开,生产服务器跑的裸 Linux 没有这层兜底,所以本机能跑的 amd64 镜像在 arm64 服务器上(或反过来)会直接 `exec format error` 崩掉。</font>

### 4.3 buildx 主线

> <font size=2>**【名词解释】<font color=red>buildx</font>(Docker buildx,Docker 构建 CLI 工具)** — Docker 官方定义是"用来跑 build 的 CLI 工具",真正执行 build 的后台引擎是 BuildKit。支持多架构 build、构建缓存共享。本课通过 `docker buildx build --platform linux/amd64` 在 Apple Silicon 上 build 出 amd64 镜像。</font>

&emsp;&emsp;Docker Engine 自带 `buildx` 工具,一条命令同时完成"在 arm64 本机用 QEMU 仿真 amd64 环境跑 build + 把结果输出到指定目的地"。在 ai-todo 项目目录里跑:

```bash
cd /path/to/ai-todo

# 先看本机 buildx 当前的 builder 状态
docker buildx ls
```

&emsp;&emsp;`docker buildx ls` 列出本机所有 buildx builder 实例及它们能 build 出哪些平台架构。第一次跑这条命令会看到一个 `default` builder 标着 `linux/arm64, linux/amd64, linux/arm/v7` 等平台——这说明 Docker Desktop 出厂自带跨架构 build 能力。如果哪天想 build linux/amd64 但 buildx ls 没看到那一行,要先创建一个支持多架构的 builder(`docker buildx create --use --name multi`)。

&emsp;&emsp;接下来 build 一份 amd64 镜像。这里有个 tag 切换要先说明:第三章本机 build 时用的是 `ai-todo:0.1`(无前缀 + 显式版本号,便于本地管理多版本),从这一章 push 到 Docker Hub 起改用 `<你的 Hub 用户名>/ai-todo:latest`(Hub 命名空间前缀 + latest,让云端 pull 命令更简洁)。这是同一份镜像的两个标签,可以共存;想保持一致把下面的 `:latest` 改成 `:0.1` 也完全可以。

```bash
# build 单平台 amd64 镜像,--load 把镜像加载到本机镜像列表
# 注意: <你的 Hub 用户名> 必须替换成你在 hub.docker.com 注册的真实账号,这是 Hub 服务端的硬性规则
docker buildx build --platform linux/amd64 \
    -t <你的 Hub 用户名>/ai-todo:latest \
    --load .
```

&emsp;&emsp;`--platform linux/amd64` 让 buildx 仿真 amd64 环境跑 Dockerfile,`--load` 让 build 完的镜像出现在本机 `docker images` 列表里方便本机验证。这条命令第一次跑会显著慢于本架构 build——QEMU 仿真 x86 指令集 +5-10 倍的开销是已知的代价。镜像名前缀 `<你的 Hub 用户名>` 是占位符,push 到 Hub 时这里必须是你真实的 Hub 账号名——前缀和登录账号不一致 Hub 服务端直接拒,4.5 节会详细展开这条规则。

&emsp;&emsp;**三平台差异提示**——上面这条 buildx 命令在 macOS Terminal 和 Linux 终端里都一样,Windows 用户在 PowerShell 里需要把行尾的 `\` 续行符换成反引号 `` ` ``,在 Git Bash 里则跟 macOS / Linux 完全一致。

### 4.4 --load 和 --push 的分工

&emsp;&emsp;buildx 输出有两个开关:`--load` 把镜像加载到本机 docker 镜像列表(供本机 docker run 用),`--push` 直接推到 Registry(本机镜像列表里看不到)。**单平台 build 下两个都能用**——本课 4.3 用 `--load` 看本机镜像,4.5 用 `docker push` 推 Hub 分两步;也可以一条命令直接 `--push` 一气呵成。

&emsp;&emsp;**真正的限制是多平台 build**——`--platform linux/arm64,linux/amd64` 一次出两份产物时 `--load` 报错,只能 `--push`。原因:本机 docker daemon 存储不支持一个 `name:tag` 挂多份架构(每个 tag 只能挂一个具体 manifest);Docker Hub 等 Registry 通过 <b>manifest list(OCI Image Index)</b>实现"一个 tag 同时挂多架构",`docker pull` 时 Registry 按客户端 CPU 自动挑对应那份返回(`python:3.12-slim` 在 Mac 拉到 arm64、在云上拉到 amd64 就是这机制)。本课只 build amd64 一份给云端拉,单平台不撞这个限制;将来要做开源跨架构分发,就走 `--push` + manifest list。

### 4.5 推到 Docker Hub

&emsp;&emsp;push 之前要 `docker login`——Docker Hub 现在用 Personal Access Token(PAT)做认证,密码登录已经被淘汰。生成 PAT 的路径是:`hub.docker.com` 登录 → 右上角头像 → Account Settings → Security → New Access Token → scope 选 Read/Write/Delete → 生成。这个 Token 只在创建时显示一次,关掉就再也看不到——一定要立即复制保存到密码管理器里。

```bash
# 用 PAT 登录(用户名是 Docker Hub 用户名,密码栏粘贴 PAT)
docker login -u <你的 Hub 用户名>

# push 到 Hub
docker push <你的 Hub 用户名>/ai-todo:latest
```

&emsp;&emsp;`docker login` 把认证凭据写到 `~/.docker/config.json`,后续 push / pull 都自动用这份凭据。push 命令把本机镜像分层上传到 Hub,已经存在的层会被服务端 skip(layer cache 跨机器复用)。

&emsp;&emsp;<b>镜像名前缀必须等于你的 Docker Hub 用户名——这是 Hub 服务端的硬性规则,不是约定俗成</b>。镜像名格式 `<用户名>/<镜像名>:<tag>`,Hub 通过前缀判断"推到哪个命名空间",前缀和你登录的账号不一致直接被 401/403 拒绝。本课件里出现的 `your-username` / `yanggggg` 都只是占位符——你跑这条命令时必须替换成自己在 hub.docker.com 注册的真实用户名(右上角头像下拉看到的那个),否则 push 会被拒。

> <font size=2>**【提示】** 这条规则只针对 Docker Hub。如果用别的 Registry(腾讯云 TCR / 阿里云 ACR / GitHub Container Registry 等),前缀规则也类似——"必须等于你在那家 Registry 里有写权限的命名空间",但具体格式各家不一样:GHCR 是 `ghcr.io/<GitHub 用户名>/<镜像名>`,腾讯云 TCR 是 `<自定义域名>/<命名空间>/<镜像名>`,等等。Docker Hub 和 GitHub 是两套完全独立的账号系统,Hub 不认 GitHub 用户名。</font>

&emsp;&emsp;实际项目里,build + push 一气呵成是更常用的写法:

```bash
# build amd64 + 直接 push,不在本机存中间产物
# 注意: <你的 Hub 用户名> 必须替换成 hub.docker.com 注册的真实账号
docker buildx build --platform linux/amd64 \
    -t <你的 Hub 用户名>/ai-todo:latest \
    --push .
```

&emsp;&emsp;这条命令完成后,在 hub.docker.com 自己的 repo 页面就能看到这份镜像,标签是 `latest`、架构是 `linux/amd64`。Hub 上显示的是压缩后体积,跟本机 `docker images` 显示的差不多,通常都在 70-75 MiB 区间。

&emsp;&emsp;<b>关于镜像 size 字段的两种统计口径</b>——`docker images` 显示的 size 是<b>解压后所有 layer 总占用空间</b>,Hub 上显示的是<b>OCI manifest 里 layer blob 压缩后总和</b>。这两个数字通常差异不大(ai-todo 这种 Python 应用约 72 MiB / 70 MiB),但如果用 `docker system df` 看本机总占用还会更大,因为它把所有共享的基础镜像层一起汇总。本课的取舍:<b>别太纠结 size 字段数字</b>,push / pull 实际传输字节看 Hub 显示的压缩值,本机磁盘占用看 `docker system df`。

### 4.6 Docker Hub 限额

&emsp;&emsp;Docker Hub 对镜像 pull 是有限额的,这点跑实验跑多了就会撞——课程实验环节如果反复 build / push / pull,限额很快用完。<b>2026 年 5 月当前的限额规则</b>:

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Docker Hub 镜像 pull 限额</font></p>
<div class="center">

| 账号类型 | 限额 | 计数维度 |
|---|---|---|
| 匿名用户(未登录) | 100 次 / 6 小时 | 按 IP |
| Personal 免费账号(已登录) | 200 次 / 6 小时 | 按账号 |
| Pro 付费账号 | 无限制 | — |

</div>

&emsp;&emsp;4.5 节本机已经 `docker login` 过,凭据存在 `~/.docker/config.json`,本机这台走的是 <b>Personal 200 次/6h</b>。注意每台机器要单独 login——腾讯云那台还没登录,默认是匿名 100 次/6h。撞限额时报 `toomanyrequests: You have reached your pull rate limit`,要么 `docker login` 翻倍到 200,要么等限额窗口滑过(<b>滑动 6 小时,不是日历日复位</b>),要么用下一节的"腾讯云内网 mirror"完全绕开 Hub 限额。

### 4.7 腾讯云内网 mirror

&emsp;&emsp;<b>前置:云端先装 Docker Engine</b>——前置课在腾讯云 CVM 上配的是 nginx + certbot + uvicorn(venv)那一套,<b>默认没有 docker</b>。下面所有命令(配 mirror、4.8 节 pull/run)都依赖云端有 docker,所以<b>第一次走到这一步</b>先在云端装一次:

```bash
# === 在本机 Mac 上跑(先 ssh 进云端)===
ssh ubuntu@腾讯云公网IP
```

```bash
# === 以下命令在腾讯云 CVM 上跑 ===
# 用腾讯云镜像源装 Docker Engine(国内拉 docker.com 的包慢且偶尔超时)

# 第一步:导入腾讯云 docker-ce 仓库的 GPG 公钥,转成二进制密钥环格式存到
# /usr/share/keyrings/,作用是让 apt 后面拉到的 docker .deb 包能通过签名校验
# (没这步 apt 会拒装"来源不可信"的包)
curl -fsSL https://mirrors.cloud.tencent.com/docker-ce/linux/ubuntu/gpg \
    | sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg

# 第二步:把腾讯云 docker-ce apt 源地址写进 /etc/apt/sources.list.d/docker.list,
# 用 signed-by= 显式绑定上一步那份公钥,告诉 apt "去这个 URL 拉 docker 包,用
# 这份 key 验签",dpkg --print-architecture 自动填本机架构(amd64/arm64),
# lsb_release -cs 自动填 Ubuntu 代号(jammy/noble 等),不用手敲对应版本字符串
echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/docker-archive-keyring.gpg] https://mirrors.cloud.tencent.com/docker-ce/linux/ubuntu $(lsb_release -cs) stable" \
    | sudo tee /etc/apt/sources.list.d/docker.list > /dev/null

sudo apt-get update -q
sudo apt-get install -y docker-ce docker-ce-cli containerd.io

# 把 ubuntu 用户加进 docker 组,后续不用每条 docker 命令都加 sudo
sudo usermod -aG docker ubuntu

# 验证 docker 版本(应该看到 29.x 以上)
docker --version
```

&emsp;&emsp;`apt-get install` 那行一次装了三个独立的包,**它们不是"装 docker"的三个等价别名,而是 docker 内部三层架构的三个组件**,各管各的:

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>docker 三件套包职责</font></p>
<div class="center">

| 包名 | 角色 | 装上后多了啥 |
|---|---|---|
| `docker-ce` | Docker 守护进程(<b>dockerd</b>) | 多一个 `dockerd` 长驻进程,通过 systemd 起来后监听 `/var/run/docker.sock`,负责镜像 / 容器 / 网络 / 卷的对象生命周期管理 |
| `docker-ce-cli` | docker 命令行客户端(<b>docker</b>) | 多一个 `docker` 命令,我们敲的 `docker ps` / `docker run` 都是它——它本身不创建容器,只把请求通过 socket 发给 dockerd |
| `containerd.io` | 底层容器运行时(<b>containerd</b> + <b>runc</b>) | 多两个二进制:`containerd`(管容器生命周期的低层服务) + `runc`(实际调内核 namespace / cgroup 创建容器进程的最底层工具) |

</div>

&emsp;&emsp;<b>调用链是单向往下的</b>:我们敲 `docker run nginx` → `docker` CLI 把请求 POST 给 `dockerd` 的 socket → `dockerd` 通过 gRPC 调 `containerd` → `containerd` 调 `runc` → `runc` 用 `clone()` 系统调用让内核 namespace + cgroup 隔离出新进程,nginx 跑起来。**三个包分别管这条链的一环**,缺一个 docker 都跑不起来。

&emsp;&emsp;<b>为什么不打成一个包</b>:历史上 docker 起家时确实是单体,后来 Docker 把"容器跑起来"这件事剥离成独立的 `containerd` 项目(2017 捐给 CNCF),k8s 早期通过 docker 调容器,后来 1.24 直接砍掉 docker 改走 containerd——这就是为什么 `containerd.io` 现在能脱离 docker 独立装。三个包独立打的好处是:① 升 daemon 不一定要升 CLI,② 跑 k8s 节点可以只装 `containerd.io` 不装 docker。

&emsp;&emsp;`usermod -aG docker ubuntu` 改组需要重新登录才生效——`exit` 退出当前 ssh 会话再 `ssh ubuntu@...` 重新进来,之后 `docker ps` 不需要 sudo 就能跑。下面 mirror 配置那段仍然带 `sudo` 因为写 `/etc/docker/daemon.json` 需要 root 权限。

&emsp;&emsp;**先讲下一步要解决的三个真实问题**——4.8 节我们要在腾讯云 CVM 上 `docker pull yanggggg/ai-todo:latest`,如果不做任何配置直接拉,会同时撞上三个坑:

- **速度慢**:Docker Hub 在国外,腾讯云走公网到 Hub 通常只有几百 KB/s 到 1-2 MB/s,一个几百 MB 的镜像要等好几分钟

- **公网流量收费**:腾讯云按量计费实例的公网下载流量是计费的,反复拉镜像每 GB 都是钱

- **撞 Hub 限额**:4.6 节讲过腾讯云那台默认匿名 100 次/6h,反复实验很容易耗光

&emsp;&emsp;<b>腾讯云的解法是内网 mirror</b>——给买了 CVM 的用户提供一个<b>内网镜像加速地址 `mirror.ccs.tencentyun.com`</b>,把它配进 docker daemon 的 `registry-mirrors` 字段,所有 `docker pull` 类的命令都先走腾讯云内网拉,内网拉不到再回源到 Docker Hub。三个问题一次性解决:<b>内网带宽通常是公网的几十倍(几十 MiB/s) + 内网流量不计费 + mirror 内置的缓存层不计入 Docker Hub 限额</b>。

&emsp;&emsp;这个 mirror **只在腾讯云 VPC 内可达**——你本机 Mac 上 `curl mirror.ccs.tencentyun.com` 是不通的,这是腾讯云给自家 CVM 用户的内部福利,不对外开放。所以下面的 mirror 配置命令仍然在腾讯云 CVM 上跑(刚才装完 docker 重新 ssh 进去那次会话里直接继续):

```bash
# === 以下命令在腾讯云 CVM 上跑 ===
# 准备 docker daemon 配置目录(可能已存在)
sudo mkdir -p /etc/docker

# 写 mirror 配置(单条 daemon.json 就够)
echo '{"registry-mirrors": ["https://mirror.ccs.tencentyun.com"]}' \
    | sudo tee /etc/docker/daemon.json

# 重启 docker daemon 让新配置生效
sudo systemctl restart docker

# 验证 mirror 配置已被 daemon 读取
docker info | grep -A 2 "Registry Mirrors"
```

&emsp;&emsp;`docker info` 输出里能看到 `Registry Mirrors: https://mirror.ccs.tencentyun.com` 这一行就说明配置生效了。下一节(4.8)在腾讯云上 `docker pull yanggggg/ai-todo:latest` 时,daemon 会自动走 mirror 拉,几十 MiB/s 的速度跟之前公网走 Docker Hub 的差距非常明显。

> <font size=2>**【提示】<b> 这个内网 mirror </b>只用于 pull**,不影响 push。本课的 push 操作是在本机 Mac 上完成的(4.5 节),走的是公网到 Docker Hub,跟腾讯云 mirror 无关。其他云厂商有类似的内网 mirror——阿里云是 `https://<你的ID>.mirror.aliyuncs.com`(需要在控制台申请),华为云是 `swr-cache.cn-north-4.myhuaweicloud.com`,配置方式跟腾讯云完全一致(改 daemon.json 的 `registry-mirrors`)。</font>

### 4.8 腾讯云拉起 + nginx 不改

&emsp;&emsp;现在 amd64 镜像已经在 Docker Hub 上,腾讯云 mirror 也配好了,把 ai-todo 在腾讯云上跑起来。这一节涉及两台机器,**每个命令块开头会标清楚"在本机跑"还是"在腾讯云跑"**,跟着标签走就不会撞坑。

&emsp;&emsp;**第一步**:在本机 Mac 上把 `.env` 文件传到云端。`.env` 含 OpenRouter API key,前置课的安全约束要求它只能放在云端 `/home/ubuntu/ai-todo/.env`,不进镜像、不进 git。

```bash
# === 在本机 Mac 上跑 ===
# 先在云端建好目标目录(否则 scp 会报路径不存在)
ssh ubuntu@腾讯云公网IP "mkdir -p /home/ubuntu/ai-todo"

# 本机的 .env 传到云端 (把 /path/to/ai-todo 换成你本机的真实路径)
scp /path/to/ai-todo/.env \
    ubuntu@腾讯云公网IP:/home/ubuntu/ai-todo/.env

# 立刻收紧文件权限,只让 owner 读写
ssh ubuntu@腾讯云公网IP "chmod 600 /home/ubuntu/ai-todo/.env"
```

&emsp;&emsp;**第二步**:ssh 进腾讯云,在云端拉镜像 + 跑容器。

```bash
# === 在本机 Mac 上跑 ===
ssh ubuntu@腾讯云公网IP
```

```bash
# === 以下命令都在腾讯云 CVM 上跑(ssh 进去之后)===
# 注意: <你的 Hub 用户名> 必须替换成 4.5 节 push 时用的真实账号
docker pull <你的 Hub 用户名>/ai-todo:latest

# 跑容器之前先建好数据卷目录,并把所有权改成跟容器内 app 用户一致
# 不做这步,容器内 app 用户写 /app/data/todos.db 会撞 Permission denied
# 详见 3.7 节末尾的提示

# 第一步:查容器内 app 用户的真实 UID/GID(本课实测 ai-todo 镜像是 1000:1000)
docker run --rm <你的 Hub 用户名>/ai-todo:latest id app

# 第二步:用查到的数字 chown 宿主机挂载目录
sudo mkdir -p /home/ubuntu/ai-todo-data
sudo chown -R 1000:1000 /home/ubuntu/ai-todo-data

docker run -d --name ai-todo \
    -p 127.0.0.1:12234:12234 \
    --env-file /home/ubuntu/ai-todo/.env \
    -v /home/ubuntu/ai-todo-data:/app/data \
    --restart unless-stopped \
    <你的 Hub 用户名>/ai-todo:latest
```

&emsp;&emsp;`docker pull` 拉镜像(走腾讯云 mirror 加速)+ `docker run` 起容器,容器把 12234 端口映射到宿主机 12234 端口,绑 loopback 不对公网开。这份 docker run 命令跟第三章本机跑的几乎完全一致,唯一区别是路径——`.env` 在 `/home/ubuntu/ai-todo/.env`、数据卷在 `/home/ubuntu/ai-todo-data`。腾讯云上 nginx 反代上游配置仍然是 `proxy_pass http://127.0.0.1:12234`,跟前置课完全相同——容器化没破坏 nginx 配置,改的只是 12234 端口背后的进程实现。

&emsp;&emsp;**`--restart unless-stopped` 是容器版的"进程守护"**——前置课的 ai-todo 是 venv 跑的 uvicorn 进程,主机重启或进程崩溃要靠 systemd ai-todo.service 拉回来。换到容器之后,这件事 docker 自己接管了:`--restart unless-stopped` 告诉 docker daemon "容器退出就重启,除非你手动 docker stop 它"。前置课的 systemd 服务可以彻底 disable 掉(已经是 `inactive (dead)`),容器化后用不到。

<style>.center{width:auto;display:table;margin-left:auto;margin-right:auto;}</style>
<p align="center"><font face="黑体" size=4>docker --restart 四个策略对照</font></p>
<div class="center">

| 策略 | 行为 | 用什么场景 |
|---|---|---|
| `no`(默认) | 容器退出就退出,不重启 | 一次性任务(比如 init container 下载完就退出) |
| `on-failure` | 只有非 0 退出码才重启 | 想区分"正常退出"和"异常崩溃",只对崩溃做守护 |
| `unless-stopped`  | 退出就重启,**除非手动 `docker stop`** | 生产环境常用——既自动恢复,又允许主动停服务时不被烦 |
| `always` | 退出就重启,主机重启 + dockerd 重启都会拉起来 | 极少用,基本场景 `unless-stopped` 都够 |

</div>

&emsp;&emsp;<b>三层守护关系要捋清</b>——前置课只有 systemd 一层,容器时代变成三层:① <b>dockerd 自己</b>靠主机 systemd 守护(apt 装 docker 时自动配好 docker.service);② <b>容器对象</b>靠 dockerd 守护(按 `--restart` 策略拉起);③ <b>应用进程</b>(容器内 PID 1 的 uvicorn)靠 dockerd 守护(容器内进程崩 → 容器退出 → docker 按策略重启容器 → uvicorn 重新起来)。也就是说,本课 `--restart unless-stopped` 一行替代了前置课"写 ai-todo.service unit 文件 + `systemctl enable` + `systemctl start`"那一整套。主机重启后,只要 dockerd 起来(systemd 保证),容器就会按这条策略自动启回来,不需要任何额外配置。

&emsp;&emsp;前置课已经把 nginx + certbot HTTPS 配好的腾讯云 CVM 上,容器跑起来之后只需要 `sudo systemctl reload nginx`(其实不 reload 也行,因为 nginx 配置一行没改),然后从本机 curl 验证:

```bash
# === 在本机 Mac 上跑(从外部公网访问云端服务)===
curl -i https://myagent-lab.online/health
```

&emsp;&emsp;`curl -i` 输出完整 HTTP 响应头 + body,我们看 `HTTP/2 200` 状态码 + body 里的 `{"status": "ok"}`。这条命令验证完整的端到端链路:本机 curl → DNS → 腾讯云 nginx 443 → 反代到 127.0.0.1:12234 → ai-todo 容器 → SQLite。返回 200 + JSON body,浏览器访问 `https://myagent-lab.online` 地址栏的 HTTPS 锁图标依然正常(前置课配的证书没动过),整条链路打通——单容器线从本机到云上的完整闭环就走通了。

---

## <center>第五章 文搜图全链路:从本机 hybrid 到生产编排</center>

&emsp;&emsp;前四章我们把 ai-todo 单容器线收口在腾讯云——一份 amd64 镜像走 Hub 中转,在云端机器上 `docker run` 起来,前置课的 nginx 反代一行不改,单进程的"打包 + 分发"叙事闭环了。但实际部署里更常见的不是单进程,而是一组互相依赖的服务。本章换主角到文搜图——5 个服务、双 Qwen3-VL 模型、GPU 推理,把 docker compose 多服务编排的完整主线一次跑通,从本机 hybrid 一直到镜像 push Hub 准备上服务器。

&emsp;&emsp;到这里我们已经会用 `docker run` 把一个服务跑起来,加上 `-p` 端口、`--env-file` 注入密钥、`-v` 卷挂载持久化、`--restart` 自愈。但是如果遇到一个复杂的项目——比如文搜图这个项目有 5 个服务:一个 FastAPI 后端、一个 Vite 前端、加上 Milvus 三件套(etcd 配置中心、minio 对象存储、Milvus 向量库本体)。如果用 `docker run` 一条一条起,5 条命令、5 套端口、5 套卷、5 套网络,谁先谁后全靠人脑记,关机重启全部手动重来。Docker Compose 就是为这种场景设计的——把这 5 个服务的描述、它们的启动顺序、它们之间的网络全部写进一份 `docker-compose.yml`,`docker compose up` 一条命令拉起全部。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/5.0-%E6%96%87%E6%90%9C%E5%9B%BE%E5%85%A8%E9%93%BE%E8%B7%AF-9485c62a.png" width=80%>
</div>

&emsp;&emsp;不过这一章我们还会撞到一个第三章 ai-todo 没遇到的问题:<b>本机的容器跑不了 GPU 加速</b>。Apple Silicon 上 Metal / MPS 加速不能透传进 Docker Desktop 的 Linux 虚拟机,本机 backend 进容器只能跑 CPU 模式,单次 Qwen3-VL embedding 推理 30-60 秒,几乎不可用。所以本章的本机部分采取 hybrid 模式——Milvus 三件套这种"基础设施"装在 docker 里省事,backend / frontend 这两个吃硬件加速的留在 host venv / npm 跑;生产服务器(下一章的 4GPU24G)是 Linux + NVIDIA GPU,这条限制天然消失,所有服务都进容器一条命令拉起。

&emsp;&emsp;这一章我们也顺便把"什么时候用 hybrid、什么时候全 docker"的三平台兼容矩阵讲清楚,这是工业部署绕不开的现实——Mac 开发、Linux 上线,容器化的边界要先看清。

> <font size=2>**【名词解释】<font color=red>hybrid</font>**(hybrid mode,混合部署模式)— 一部分服务进容器、一部分留在宿主机直接跑的部署形态。本课的 hybrid 具体是 Milvus 三件套(etcd / minio / standalone)装在 docker 里(基础设施类服务,版本一致性靠容器保证),backend / frontend 这两个吃硬件加速的留在 host venv / npm 跑(走 macOS 的 MPS 加速,单次推理 1-3 秒)。这种模式只在本机开发场景需要,根因是宿主机的 GPU 加速(MPS / CUDA)不能完整透传进容器 VM;到了生产服务器(Linux + NVIDIA GPU),CUDA 能直接透传进容器,所有服务都能一条命令进 docker,hybrid 自然消失。</font>

### 5.1 换主角:文搜图 5 服务速览

&emsp;&emsp;前三章我们一直跟 ai-todo 打交道——一个 FastAPI 服务、一份 SQLite、一个对外端口。SQLite 内嵌在进程里,不需要独立的数据库服务,所以 compose 的核心问题(启动顺序、健康检查、服务间网络、跨服务依赖)在 ai-todo 里都不出现。学一个工具最好的方式是用真正需要它的项目,文搜图就是这样的项目——它本身就是 5 服务的多模态向量检索系统,5 个服务里任何一个挂了整体不工作,刚好把 compose 要解决的所有问题一次摆齐。

> <font size=2>**【名词解释】<font color=red>Qwen3-VL</font>**(Qwen3 Vision-Language,通义千问视觉语言模型)— 阿里通义千问开源的视觉语言模型家族,本课用到两个 2B 参数规模的变体:<b>Qwen3-VL-Embedding-2B</b> 把图片和文本编码到同一个 512 维向量空间;<b>Qwen3-VL-Reranker-2B</b> 对粗筛的候选图片做精排,提升 top-N 准确率。两个模型权重共约 8 GiB,本课从 ModelScope 拉取。</font>

> <font size=2>**【名词解释】<font color=red>Milvus</font>**(Milvus,向量数据库)— 开源向量数据库,本课用 standalone(单机)部署模式,实际是三个进程联动。**milvus-standalone** 是本体,负责把向量装进索引(IVF / HNSW)、跑相似度检索;**etcd** 是元数据存储,记的是"集合叫什么名 / 有哪些字段 / 用了哪种索引 / 分片路由表"这类<b>结构性描述信息</b>(KB 级,改一次写一行),典型例子:`collection: t2i_images, dim: 512, metric: IP, index_type: HNSW`;**minio** 是 S3 兼容的对象存储,放的是<b>真正的向量数据本体和大块二进制</b>(MB / GB 级),典型例子:512 维 fp32 向量的 binary segment 文件、写入日志的 WAL 段、批量插入的 binlog。三件套各管一摊:元数据归 etcd,大文件归 minio,索引和查询归 standalone,三个都 healthy 才能让 standalone 工作。</font>

&emsp;&emsp;5 个服务各自的职责和端口如下面这张表所示。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>文搜图 5 个服务的角色</font></p>
<div class="center">

| 服务名 | 职责 | 监听端口 | 镜像来源 |
|---|---|---|---|
| etcd | 元数据存储(KV 数据库,存集合 schema / 索引参数 / 分片路由) | 2379 | quay.io/coreos/etcd:v3.5.5 |
| minio | 对象存储(S3 兼容,存向量 binary segment / WAL / binlog) | 9000 / 9001 | minio/minio:RELEASE.2023-03-20T20-16-18Z |
| milvus-standalone | 向量库本体(向量索引 + 检索引擎) | 19530 / 9091 | milvusdb/milvus:v2.3.21 |
| backend | FastAPI + Qwen3-VL 双模型 | 3001 | 自 build(`yanggggg/wensoutu-backend:0.1`) |
| frontend | Vite 构建产物 + nginx 静态服务 | 80 | 自 build(`yanggggg/wensoutu-frontend:0.1`) |

</div>

&emsp;&emsp;关键观察——<b>Milvus 不是单一服务,而是三件套</b>。三个组件各管一类数据:**etcd** 管元数据(集合 schema / 字段类型 / 索引参数 / 分片路由,KB 级,改一次写一行,典型例子 `collection: t2i_images, dim: 512, metric: IP, index_type: HNSW`)、**minio** 管对象存储(向量 binary segment 文件 / WAL 段 / binlog,MB / GB 级,本课 26 张图编完向量后这里会落约 100 MB 的 binary segment)、**standalone** 是 Milvus 本体进程(把向量装进 HNSW 索引、接收 SDK 的检索请求)。三个都必须 healthy 才能让 standalone 工作。这种"一个组件靠两个底层组件"的结构,正是 compose `depends_on` 配合 `healthcheck` 要解决的问题。

&emsp;&emsp;backend 服务承担的事情有两件:一是把 26 张演示图通过 Qwen3-VL Embedding 编码成 512 维向量存进 Milvus,二是接收用户的文字查询,把文字 embedding 出来跟 Milvus 里的图片向量做相似度检索,加上 Reranker 精排,最后返回 top-N 结果。frontend 不只是显示页面——它的容器内部跑了一份 nginx,把 `/api/` 反代到 backend、把 `/static/` 反代到 backend 的静态目录,自身只对外提供 vite 编译出来的 SPA 静态资源,我们后面 5.7 节会展开这份 nginx.conf。

### 5.2 平台边界:三平台兼容矩阵

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/5.2-%E4%B8%89%E5%B9%B3%E5%8F%B0%E5%85%BC%E5%AE%B9%E7%9F%A9%E9%98%B5-e25fc5af.png" width=80%>
</div>

&emsp;&emsp;到这里我们要先把一个工程现实摆出来:**AI项目容器化能不能"全放进docker"取决于宿主机的硬件加速能不能透传进容器**。这个问题在 ai-todo 上不存在(CPU 跑 SQLite + LangGraph,容器 CPU 完全够用),但文搜图要跑 Qwen3-VL 模型推理,CPU 模式单次 embedding 30-60 秒、用户根本等不及,必须吃硬件加速——这就跟操作系统厂商的生态决策直接挂钩了。

&emsp;&emsp;<b>苹果把 Metal 锁死在 macOS 原生进程</b>——Docker Desktop 在 Mac 上的实现底层跑了一个 Linux 虚拟机(第一章 1.4 节"容器 vs 虚拟机 vs 物理服务器"我们讲过这件事),容器跑在这个 VM 里,而 Apple Silicon 的 Metal / MPS 加速被苹果设计为只能在 macOS 原生进程里调用,VM 拿不到。<b>NVIDIA 和微软合作做了 WSL2 CUDA passthrough</b>——Windows 上的 Docker Desktop 走 WSL2 backend,WSL2 里的 Linux 进程可以拿到宿主 NVIDIA GPU,所以 Windows + NVIDIA 卡能"全 docker"。<b>Linux 是 docker 原生宿主</b>,nvidia-container-toolkit 或 Docker 29+ 内置 CDI 都能让容器直接访问宿主 GPU。

> <font size=2>**【名词解释】<font color=red>Metal</font>**(Metal,苹果原生 GPU 框架)— Apple 在 macOS / iOS 上的底层 GPU 加速 API,角色类似 NVIDIA 的 CUDA,直接对接 Apple Silicon GPU 硬件。本课的 MPS 是基于 Metal 的封装层——PyTorch 走 `torch.backends.mps` 后端,底下还是在调 Metal。<b>真正被锁死在 macOS 原生进程的是 Metal 这个底层框架</b>,MPS 作为上层封装跟着一起被挡在 Docker Desktop 的 Linux VM 外面。</font>

> <font size=2>**【名词解释】<font color=red>MPS</font>**(Metal Performance Shaders,Metal 性能着色器)— Apple 提供的一组基于 Metal 的高性能 GPU 算子库(矩阵乘 / 卷积 / 归约等),PyTorch 通过 `torch.backends.mps` 后端调用 Apple Silicon 的 GPU。MPS 跟着底层 Metal 一起不能透传进 Docker Desktop 的 Linux VM,这是本课本机 backend 不进容器的根本原因。</font>

&emsp;&emsp;矩阵里第一行就是本课开发机的情形——Mac Apple Silicon。我们走 hybrid,Milvus 三件套进 docker,backend / frontend 留在 host;矩阵最后一行(Linux + NVIDIA GPU)就是下一章生产服务器 4GPU24G 的情形,所有服务全 docker。开发跟生产的差异落在两点:hybrid 模式下 backend 用 host venv 启动、连接 `localhost:19530` 上的 Milvus;全 docker 模式下 backend 进容器、连接 `milvus-standalone:19530`(服务名当 hostname,compose 内部网络解析)。这条边界讲清楚了,接下来 5.3 节先把 compose yaml 的骨架过一遍,5.4 节再把本机 hybrid 真跑起来。

### 5.3 Compose yaml 结构速览

&emsp;&emsp;第一次看 `docker-compose.yml` 容易被"看起来缩进很复杂的 yaml"吓住,但其实顶层只有 4 个大块——`name` / `services` / `networks` / `volumes`,每个 service 里也是固定那十来个字段。先把骨架过一遍,5.4 节看本机的 `docker-compose.deps.yml` 实际怎么填这些字段。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/5.3-Compose-yaml%E7%BB%93%E6%9E%84%E9%80%9F%E8%A7%88-a3bb3099.png" width=80%>
</div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>docker-compose.yml 顶层结构</font></p>
<div class="center">

| 顶层字段 | 干啥 | 必填? |
|---|---|---|
| `name:` | compose 项目名,docker 用它管理容器组(默认从目录名拿,路径含中文必须显式声明) | 推荐写 |
| `services:` | 所有服务的定义,下面每个键是一个服务名,服务名同时也是 docker 内部网络的 hostname | 必填 |
| `networks:` | 服务间网络拓扑,默认 compose 会自建一个 bridge 网络,所有 service 自动加入 | 可省 |
| `volumes:` | 命名卷声明,服务之间共享数据时用 | 可省 |

</div>

&emsp;&emsp;`services:` 下每个服务自己又有十来个字段——常用的就那些,我们也先列出来。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>services 下单个服务的核心字段</font></p>
<div class="center">

| 字段 | 干啥 | 典型用法 |
|---|---|---|
| `image:` | 用哪个镜像(从 Hub / 私有 registry 拉) | `image: yanggggg/wensoutu-backend:0.1` |
| `build:` | 不用现成镜像,从 Dockerfile 现 build(跟 `image` 二选一) | `build: ./backend` |
| `container_name:` | 给容器起名,`docker ps` 看到的就是这个 | `container_name: wensoutu-backend` |
| `ports:` | 端口映射,宿主机:容器 | `ports: ["127.0.0.1:3001:3001"]` |
| `environment:` | 环境变量,等价于 `docker run` 的 `--env` | `environment: [MILVUS_URI=...]` |
| `volumes:` | 卷挂载,宿主机路径:容器路径 | `volumes: ["./volumes/models:/app/models"]` |
| `depends_on:` | 启动顺序依赖(可加 condition 等到 healthy 再起) | `depends_on: {milvus: {condition: service_healthy}}` |
| `healthcheck:` | 健康探针(test / interval / timeout / retries / start_period) | `healthcheck: {test: [CMD, curl, ...]}` |
| `restart:` | 退出后重启策略 | `restart: unless-stopped` |
| `networks:` | 这个服务挂在哪些网络上 | `networks: [wensoutu]` |

</div>

> 提示:`services` 下面缩进 2 空格表示一个服务名,服务名再缩进 2 空格(共 4 空格)就是这个服务的字段。yaml 严格缩进敏感,缩错 1 空格 `docker compose config` 就报 `mapping values are not allowed here`。第一次写 compose 用编辑器自带的 yaml 插件(VS Code / JetBrains 都有内置)就能避开这类报错。

&emsp;&emsp;骨架过完了,下一节看本机 hybrid 用的 `docker-compose.deps.yml` 实际填的内容——5 个服务只起 Milvus 三件套这 3 个。

### 5.4 本机 hybrid 跑通

&emsp;&emsp;接续 5.3 的骨架,我们看本机 hybrid 实际怎么填这些字段。hybrid 跑通的三步:① docker 起 Milvus 三件套(5.4.1);② host venv 跑 backend(5.4.2);③ 另开终端 host 跑 frontend(5.4.3)。三步走完,浏览器开 `http://localhost:3000/` 输入"红色的包"能返回 6 张图,就算本机 hybrid 通了。

#### 5.4.1 起 Milvus 三件套(docker)

&emsp;&emsp;这份编排文件是 `docker-compose.deps.yml`——专门只声明三件套,跟生产用的 `docker-compose.prod.yml` 是两份不同文件,前者本机用、后者服务器用。

```yaml
# docker-compose.deps.yml (核心片段, 完整版见仓库)
# 只跑 milvus 三件套, backend / frontend 都在 Mac host 上跑
name: text2image-deps

services:
    etcd:
        container_name: t2i-etcd
        image: quay.io/coreos/etcd:v3.5.5
        volumes:
            - ./volumes/etcd:/etcd
        healthcheck:
            test: ["CMD", "etcdctl", "endpoint", "health"]
            interval: 30s
        networks:
            - milvus

    minio:
        container_name: t2i-minio
        image: minio/minio:RELEASE.2023-03-20T20-16-18Z
        ports:
            - "9001:9001"
            - "9000:9000"
        volumes:
            - ./volumes/minio:/minio_data
        healthcheck:
            test: ["CMD", "curl", "-f", "http://localhost:9000/minio/health/live"]
        networks:
            - milvus

    milvus-standalone:
        container_name: t2i-milvus
        image: milvusdb/milvus:v2.3.21
        command: ["milvus", "run", "standalone"]
        environment:
            ETCD_ENDPOINTS: etcd:2379             # 服务名当 hostname
            MINIO_ADDRESS: minio:9000             # compose 内部网络解析
        volumes:
            - ./volumes/milvus:/var/lib/milvus
        healthcheck:
            test: ["CMD", "curl", "-f", "http://localhost:9091/healthz"]
            start_period: 90s                     # milvus 启动慢,给 90 秒预热窗口
        ports:
            - "19530:19530"                       # ← host backend 通过这个端口连上来
            - "9091:9091"
        depends_on:
            etcd:
                condition: service_healthy        # 等 etcd healthy 再启
            minio:
                condition: service_healthy        # 等 minio healthy 再启
        networks:
            - milvus

# 顶级 networks 声明: 三个服务都挂在这张网络上互相寻址
networks:
    milvus:
        name: t2i-milvus                          # 显式指定网络名, docker network ls 看到 t2i-milvus
```

&emsp;&emsp;这份 yaml 把"复杂的基础设施"扔进 docker——不用 apt 装 etcd / minio、不用编译 Milvus,一条命令全部就位。三个细节值得留意:① `depends_on` + `service_healthy` 让 standalone 等 etcd / minio 都 healthy 再起,standalone 启动慢,`start_period: 90s` 给 90 秒预热窗口;② `ETCD_ENDPOINTS: etcd:2379` 这种"服务名当 hostname"是 compose 的第一个红利——不用配 IP,内部 DNS 自动解析;③ 网络名 `t2i-milvus` 是顶级 `networks` 段显式 `name: t2i-milvus` 指定的,有 `name:` 字段时 compose 直接用给定名而不走"项目名_网络声明名"自动拼接。

&emsp;&emsp;三平台起命令略有差异,分别给:

```bash
# macOS / Linux —— 起 Milvus 三件套
cd /path/to/text2image_search
docker compose -f docker-compose.deps.yml up -d

# 等所有服务 healthy(初次启动等 standalone 起来约 60-90 秒)
docker compose -f docker-compose.deps.yml ps
```

&emsp;&emsp;`up -d` 跑完几乎立即返回,但<b>三件套真正 healthy 还要等约 60-90 秒</b>——`docker compose ps` 看到所有服务 STATUS 列从 `(health: starting)` 变 `(healthy)` 才算就绪。Windows PowerShell 命令一致,只是路径换 Windows 风格:

```powershell
# Windows PowerShell —— 命令一致, 路径用 Windows 风格
cd D:\projects\text2image_search
docker compose -f docker-compose.deps.yml up -d
docker compose -f docker-compose.deps.yml ps
```

&emsp;&emsp;成功标准:`docker compose ps` 三行全 `(healthy)`,一份 yaml + 一条命令完成"装数据库 + 装对象存储"两件事。

#### 5.4.2 起 backend(host venv,吃 MPS 加速)

&emsp;&emsp;Milvus 三件套就绪之后,backend 在 host venv 里跑——`backend/requirements.txt` 声明了 torch / transformers / fastapi / pymilvus 等几十个包,Mac M 系列上 `pip install torch` 会自动装 MPS 版本。首次跑要先在项目根目录建 venv + 装依赖:

```bash
# macOS / Linux —— 首次创建 venv + 装依赖 (5-10 分钟,torch / transformers 比较大)
cd /path/to/text2image_search
python3 -m venv backend/.venv
backend/.venv/bin/pip install --upgrade pip
backend/.venv/bin/pip install -r backend/requirements.txt

# 装好之后启动 backend (吃 MPS 加速)
backend/.venv/bin/python -m uvicorn backend.main:app --host 0.0.0.0 --port 3001
```

```powershell
# Windows PowerShell —— 首次创建 venv + 装依赖 (5-10 分钟)
cd D:\projects\text2image_search
python -m venv backend\.venv
backend\.venv\Scripts\python.exe -m pip install --upgrade pip
backend\.venv\Scripts\python.exe -m pip install -r backend\requirements.txt

# 装好之后启动 backend
backend\.venv\Scripts\python.exe -m uvicorn backend.main:app --host 0.0.0.0 --port 3001
```

&emsp;&emsp;成功标准:stdout 打印"`[完成] 检索引擎和Agent就绪`"——`main.py` 的 lifespan 段先 `retrieval_engine.initialize()` 加载 Qwen3-VL 双模型 + 索引,再 `agent_manager.initialize()` 拉起 Agent(MPS 首启 30-60 秒,命中本机 ModelScope 缓存再次启动 10-20 秒)。**backend 在 host venv 跑、不进容器是关键**——PyTorch 拿到 MPS 加速,单次 embedding 1-3 秒;backend 通过 `MILVUS_URI=http://localhost:19530` 连到上一步 docker 里的 Milvus。

#### 5.4.3 起 frontend(host vite dev server)

```bash
# macOS / Linux —— frontend 另开一个新终端
cd /path/to/text2image_search/frontend
npm install        # 首次跑: 装 package.json 声明的依赖 (React / Vite / lucide-react 等, 约 1-2 分钟)
npm run dev        # vite dev server, 端口 3000, 自带 /api 代理到 backend:3001
```

```powershell
# Windows PowerShell —— frontend 同样另开终端
cd D:\projects\text2image_search\frontend
npm install
npm run dev
```

&emsp;&emsp;成功标准:vite 起 dev server 监听 `localhost:3000`,自带 `/api` 代理到 `localhost:3001`(hybrid 模式 backend 在 host,见 `frontend/vite.config.ts`);浏览器打开输入"红色的包",vite 把请求转给 backend,返回 6 张图。hybrid 三件齐:基础设施进 docker、backend 吃 MPS、frontend 走 vite hot reload。本机这一步走通了,后面所有 docker 化的步骤都是为了"在另一台机器(生产服务器)上重现这套"。

### 5.5 backend 生产 Dockerfile.gpu

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/5.5-GPU%E5%A4%9A%E9%98%B6%E6%AE%B5Dockerfile-c18968e9.png" width=80%>
</div>

&emsp;&emsp;hybrid 跑通了开发态,但开发态不能直接交付——生产服务器是 Linux + NVIDIA,要把 backend 进容器跑,这就需要一份生产 Dockerfile。文搜图 backend 的 Dockerfile 是<b>多阶段构建</b>——builder 阶段装依赖、runtime 阶段只拷 venv,把镜像做小;同时 builder 用完整 `python:3.11`(自带 build-essential),runtime 切到 `python:3.11-slim` 减小体积,这是 Python 生产镜像的标准模式。

> <font size=2>**【名词解释】<font color=red>Multi-stage Build</font>**(多阶段构建)+ <font color=red>CUDA Wheel</font>(CUDA 版预编译包)— ① <b>Multi-stage Build</b>:Dockerfile 里用多个 `FROM ... AS xxx` 段,builder 阶段装依赖 / 编译,runtime 阶段只 COPY 最终产物。本课 backend Dockerfile.gpu 用这个模式把 5 GB 镜像(含 cu128 libs)做到最小化——builder 完整 python:3.11,runtime 切到 python:3.11-slim。② <b>CUDA Wheel</b>:PyTorch 官方 release 的 GPU 版预编译包,wheel 文件名带 `+cu128` 后缀(对应 CUDA 12.8),自带 cuDNN / cuBLAS / CUDA runtime 等 NVIDIA 用户态库,容器只需要宿主机有兼容版本的 NVIDIA driver 就能跑,不需要装 CUDA Toolkit。</font>

&emsp;&emsp;这份 Dockerfile 我们分成 builder 段和 runtime 段两块看。先看 builder 段——任务是装好所有 Python 依赖,把 venv 留在 `/opt/venv`。

```dockerfile
# backend/Dockerfile.gpu —— builder 段
# 用 python:3.11 (非 slim) 自带 build-essential, 装 native 扩展时不缺工具
FROM python:3.11 AS builder

ENV PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple \
    PIP_DISABLE_PIP_VERSION_CHECK=1 \
    PIP_NO_CACHE_DIR=1 \
    PYTHONDONTWRITEBYTECODE=1

WORKDIR /build
COPY requirements.txt .

# 先建 venv, 装 CUDA 版 torch 走 pytorch.org cu128 专用 index (不走清华镜像)
# pip 从 cu128 index 自动选 2.9.0+cu128 wheel, 版本号不需要带 +cu128 后缀
# 其他依赖回 PyPI 清华镜像加速 (跟 cu128 index 无冲突)
RUN python -m venv /opt/venv \
    && /opt/venv/bin/pip install --upgrade pip \
    && /opt/venv/bin/pip install \
        --index-url https://download.pytorch.org/whl/cu128 \
        torch==2.9.0 torchvision==0.24.0 \
    && grep -v -E '^(torch|torchvision)==' requirements.txt > /tmp/req-no-torch.txt \
    && /opt/venv/bin/pip install -r /tmp/req-no-torch.txt
```

&emsp;&emsp;builder 段实现的是 PyTorch GPU 版本 + 业务依赖的隔离安装。关键点是 torch 必须走 `https://download.pytorch.org/whl/cu128` 这个专用 index——清华镜像没有 cu128 GPU 版本(清华只镜像 PyPI 公开包,PyTorch 的 cu12x wheel 只在 pytorch.org 上)。<b>如果误用清华镜像装 torch,会拉到 CPU 版本</b>(`torch-2.x.x-cp311-cp311-linux_x86_64.whl`,文件名无 `+cu128` 后缀),容器跑起来时 `torch.cuda.is_available()` 返回 `False`,推理自动回落到 CPU 模式,单次 embedding 从 1-2 秒拖到 30-60 秒,这是文搜图最隐蔽的 GPU 失效坑。流程作用上,builder 装完后所有 PyTorch + NVIDIA 用户态库(libcudnn ~1 GB、libcublas ~500 MB)都在 `/opt/venv/lib/python3.11/site-packages/torch/lib/` 里。项目意义是 cu128 wheel 自带 CUDA runtime,容器只需要宿主机有兼容版本的 NVIDIA driver(580.x 支持 CUDA 13.0,任何 cu12x 都能跑)就能调 GPU,**不需要装 CUDA Toolkit**——这是 torch 2.x 工业生产的标准做法。

&emsp;&emsp;接下来看 runtime 段——任务是把 builder 的 venv 拷过来,切到 slim 基镜像,加上 healthcheck 和非 root 用户。

```dockerfile
# backend/Dockerfile.gpu —— runtime 段
FROM python:3.11-slim AS runtime

# curl 给 HEALTHCHECK 用; libgomp1 给 numpy/scikit 之类的 OpenMP 用
RUN apt-get update \
    && apt-get install -y --no-install-recommends curl libgomp1 \
    && rm -rf /var/lib/apt/lists/*

# 非 root 用户 app, 跟 ai-todo Dockerfile 同模式
RUN groupadd --system app \
    && useradd --system --gid app --create-home --shell /usr/sbin/nologin app

# builder 的 venv 整个拷过来 (含 cu128 libs)
COPY --from=builder /opt/venv /opt/venv

ENV PATH="/opt/venv/bin:${PATH}" \
    PYTHONUNBUFFERED=1 \
    MODELSCOPE_CACHE=/app/backend/data/models \
    IMAGE_DIR=/app/backend/data/images
# 节选: 实际 Dockerfile.gpu 还有 PYTHONDONTWRITEBYTECODE=1 (不写 .pyc 缓存),
# 本节为节奏简洁只列核心四项,完整版见仓库 backend/Dockerfile.gpu

WORKDIR /app
RUN mkdir -p /app/backend/data/models /app/backend/data/images \
                 /app/backend/data/caption_cache /app/backend/data/uploads \
    && chown -R app:app /app

# 应用代码 (.dockerignore 排除 .venv / data / __pycache__)
COPY --chown=app:app . /app/backend
USER app

EXPOSE 3001

# start_period 给 1800 秒 (30 分钟): 首次启动从 ModelScope 下两个模型再加载
HEALTHCHECK --interval=30s --timeout=10s --start-period=1800s --retries=5 \
    CMD curl -fsS http://127.0.0.1:3001/api/health || exit 1

CMD ["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "3001"]
```

&emsp;&emsp;runtime 段三个工业级细节值得留意:① <b>HEALTHCHECK 的 `start_period=1800s`</b>(30 分钟)是给首次模型下载预留的窗口——不预先传模型时 backend 首启要从 ModelScope 下 8 GiB,期间 `docker compose ps` 一直显示 `(unhealthy)`,这不是出错而是模型还在下;② <b>非 root 用户 app</b>跟第三章 ai-todo Dockerfile 同模式,容器内进程不以 root 跑;③ <b>`MODELSCOPE_CACHE=/app/backend/data/models`</b> 让 ModelScope 库把模型落到这个目录,配合后面的 bind mount 实现"模型走卷不打镜像"。

> 提示:`backend/core/retrieval.py` 里 `embedding.cpu().numpy()` 必须先 `.float()` 转 fp32 再转 numpy——**Mac MPS 上 Qwen3-VL 输出默认是 fp32,直接 `.numpy()` 没问题**;**CUDA 上 Qwen3-VL 权重默认 bf16,输出 tensor 也是 bf16,`.numpy()` 直接报 `TypeError: Got unsupported ScalarType BFloat16`**(numpy 没原生 bf16 dtype)。修法:`embedding.float().cpu().numpy()`,先 `.float()` 强制 fp32 再 `.numpy()`。这是<b>本机 Mac 能跑、Linux GPU 必崩</b>的隐蔽类坑,跟第三章 UID 999 vs 1000 是同类——开发环境跟生产环境的硬件差异导致同样的代码行为不同。

&emsp;&emsp;build 出来的镜像约 5 GB,大头是 cu128 wheel 自带的 NVIDIA 用户态库(libcudnn ~1 GB、libcublas ~500 MB、其他 CUDA runtime .so 共 2 GB,torch 自身 Python 代码只 ~400 MB)。5 GB 在工业 GPU 应用里是正常体量,vLLM、Ollama、PyTorch Lightning 等生产镜像都在 4-8 GB 区间;如果想进一步压缩可走 `nvidia/cuda` 系列基镜像,本课用 `python:3.11-slim` + cu128 wheel 可读性最好。

### 5.6 模型权重不打镜像:工业标准

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/5.6-%E6%A8%A1%E5%9E%8B%E8%B5%B0%E5%8D%B7%E4%B8%8D%E6%89%93%E9%95%9C%E5%83%8F-5c287cb4.png" width=80%>
</div>

&emsp;&emsp;5 GB 的 backend 镜像里没有 Qwen3-VL 模型权重——两个 2B 模型共 8 GiB(`backend/data/models/` 下两个 `model.safetensors` 各 4 GiB),如果打进镜像,backend 就变 13 GB,每次 push / pull 浪费 8 GiB 流量,且模型升一版就要重 build 重 push。

&emsp;&emsp;<b>模型走 volume 是工业标准</b>——vLLM、Ollama 官方 Dockerfile 都这么做。背后的判断逻辑:① 镜像是不可变代码制品、模型权重是数据,版本节奏不同;② 私有微调权重不该出现在公开 registry,push 到 Hub 就把权重公开了;③ 多个容器实例可共享同一卷,横向扩展 4 副本共用一份 8 GiB 模型卷。具体到代码层面,`config.py` 里 `MODEL_CACHE_DIR = DATA_DIR / "models"`、`Dockerfile.gpu` 里 `ENV MODELSCOPE_CACHE=/app/backend/data/models`、`docker-compose.prod.yml` 里 `./volumes/models:/app/backend/data/models` bind mount——三处呼应,容器跑起来时 ModelScope 库看到这个目录已有模型就直接加载,否则走在线下载。这个"卷有就用、没有就下"的状态机是第六章 6.3 节"上传模型"的核心逻辑。

### 5.7 frontend 生产 Dockerfile + nginx.conf

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/5.7-frontend%20%E7%94%9F%E4%BA%A7%20Dockerfile%20%2B%20nginx.conf-f2f12b0c.png" width=80%>
</div>

&emsp;&emsp;frontend 镜像比 backend 简单——80 MB 上下、build 2-3 分钟,但有一个 backend 没有的<b>双重身份</b>:既要对外提供 vite 编译出的静态 SPA 资源,也要内置一份 nginx 把 `/api/` 反代到 backend 容器。所以这份 Dockerfile 也是多阶段——Stage 1 用 `node:20-alpine` 跑 vite build,Stage 2 用 `nginx:1.27-alpine` 跑静态服务。

```dockerfile
# frontend/Dockerfile —— 两阶段构建
FROM node:20-alpine AS builder
WORKDIR /app
RUN npm config set registry https://registry.npmmirror.com    # 国内 npm 镜像加速
COPY package.json package-lock.json ./
RUN npm ci --no-audit --no-fund                                # 装依赖
COPY . .
RUN npm run build                                              # vite build, 输出到 /app/build

FROM nginx:1.27-alpine AS runtime
COPY nginx.conf /etc/nginx/conf.d/default.conf                 # 覆盖默认 site
COPY --from=builder /app/build /usr/share/nginx/html           # vite 产物丢到 nginx web root
EXPOSE 80
HEALTHCHECK --interval=30s --timeout=5s --start-period=10s --retries=3 \
    CMD wget --quiet --tries=1 --spider http://127.0.0.1/ || exit 1
```

&emsp;&emsp;Stage 1 跑 vite build,Stage 2 用 nginx alpine 跑静态服务,最终镜像约 80 MB,部署到任何机器都不用单独 `apt install nginx`。镜像里 `nginx.conf` 配置三个职责:`location /` 兜底到 `index.html`(SPA 路由),`location /api/` 反代到 `backend:3001`(关 buffering 让 SSE 长连透传),`location /static/` 反代后端静态图片。

```nginx
# frontend/nginx.conf (核心片段)
server {
    listen 80;
    root /usr/share/nginx/html;

    location / {
        try_files $uri $uri/ /index.html;       # SPA 路由兜底
    }
    location /api/ {
        proxy_pass http://backend:3001;         # 服务名当 hostname (compose 内部网络)
        proxy_buffering off;                    # SSE 长连必须关 buffering
        proxy_read_timeout 600s;
    }
    location /static/ {
        proxy_pass http://backend:3001;         # 后端 /static 静态图片
    }
}
```

> 提示:frontend 容器内 nginx 启动时如果 backend 容器还没在 compose 网络注册 hostname,nginx 会直接 `[emerg] host not found in upstream "backend"` 然后崩溃重启,容器进入 `Restarting` 死循环。这跟 `depends_on: condition: service_started` 并不冲突——`service_started` 只等 backend 容器<b>进程</b>起来,不等 docker 网络层面的 hostname 注册。生产推荐修法是在 `nginx.conf` 顶部加 `resolver 127.0.0.11 valid=10s;`,再把 `proxy_pass http://backend:3001` 改成用变量 `set $backend http://backend:3001; proxy_pass $backend;`,让 nginx 延迟到首个请求时再解析 hostname,启动期不直接崩。

### 5.8 完整 docker-compose.prod.yml

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/5.8-Compose%E4%BA%94%E6%9C%8D%E5%8A%A1%E7%BC%96%E6%8E%92-86b826a4.png" width=80%>
</div>

&emsp;&emsp;backend / frontend 的 Dockerfile 写完后,把"5 服务怎么连"全部落在 `docker-compose.prod.yml`——`image:` 指向 Hub,`deploy.resources.reservations.devices` 声明 GPU,跟本机 `deps.yml` 是两份隔离文件。先看 backend 这段:

```yaml
# docker-compose.prod.yml —— backend 服务 (核心生产配置)
services:
    backend:
        container_name: wensoutu-backend
        image: ${DOCKER_HUB_USERNAME:-yanggggg}/wensoutu-backend:${IMAGE_TAG:-0.1}
        ports:
            - "127.0.0.1:${BACKEND_HOST_PORT:-3001}:3001"   # host 端口变量化, 撞占用改 .env.prod 一行
        environment:
            MILVUS_URI: http://milvus-standalone:19530    # 服务名当 hostname
            MODELSCOPE_CACHE: /app/backend/data/models    # 跟 Dockerfile ENV 对齐
            IMAGE_DIR: /app/backend/data/images
        volumes:
            - ./volumes/models:/app/backend/data/models   # 8 GiB 模型卷
            - ./volumes/images:/app/backend/data/images
        depends_on:
            milvus-standalone:
                condition: service_healthy      # 等 milvus healthy 再启 backend
        deploy:
            resources:
                reservations:
                    devices:
                        - driver: nvidia
                          count: 1              # 分配 1 张 GPU, 服务器有 4 张可弹
                          capabilities: [gpu]
        networks:
            - wensoutu
        restart: unless-stopped
```

&emsp;&emsp;关键点三个:① `image:` 用环境变量解析,生产服务器 `docker compose pull` 直接走 Hub;② host 侧端口走 `${BACKEND_HOST_PORT:-3001}` 变量,默认 `127.0.0.1:3001` 只绑 loopback 给宿主机 curl 调试,撞到端口被占时(企业共用服务器常见)在 `.env.prod` 加 `BACKEND_HOST_PORT=3011` 换端口,容器内 FastAPI 仍监听 3001、frontend 走服务名 `backend:3001` 不受影响;③ `deploy.resources.reservations.devices` 是 compose v3+ 的 GPU 声明,`count: 1` 让 daemon 弹性挑张可用卡。

```yaml
# docker-compose.prod.yml —— frontend 服务 + 顶层 networks
    frontend:
        container_name: wensoutu-frontend
        image: ${DOCKER_HUB_USERNAME:-yanggggg}/wensoutu-frontend:${IMAGE_TAG:-0.1}
        ports:
            - "127.0.0.1:3000:80"               # 宿主机 3000 → 容器 80 (nginx)
        depends_on:
            - backend
        networks:
            - wensoutu

networks:
    wensoutu:
        name: wensoutu                          # 显式 name, docker network ls 看到 wensoutu
```

> 提示:`deploy.resources.reservations.devices` 本机 Docker Desktop 不识别——Mac / Windows 跑这份 compose 会报 `could not select device driver "nvidia"`,因为 Docker Desktop VM 里没装 nvidia runtime。所以本机用 `deps.yml`(只有三件套,没 GPU),生产服务器用 `prod.yml`(5 服务 + GPU),两份隔离不互相 fallback。

> <font size=2>**【名词解释】<font color=red>healthcheck</font>**(healthcheck,健康检查)— compose / Dockerfile 都支持的健康探针配置,容器内周期性跑一条命令,根据退出码决定容器状态是 healthy / unhealthy / starting。Milvus 三件套各自用 etcdctl / curl 作为健康探针,backend 用 `curl /api/health` 作为探针。</font>

&emsp;&emsp;到这里把<b>容器持久化的通用规则</b>沉淀一下,因为本节 yaml 里 etcd / minio / milvus / backend 这几个服务都挂了 `./volumes/xxx`,容易让人以为"数据库类容器才需要 volume"。<b>不是这样</b>——任何容器的可写层都是 ephemeral 的,`docker rm` 一删整层数据就没了,<b>所有需要在容器重建后还活着的数据都必须走 volume</b>。三件套 yaml 里这三行就是同一条规则的三次应用:`./volumes/etcd:/etcd` 把 etcd 的元数据 KV 落宿主机、`./volumes/minio:/minio_data` 把 minio 的对象数据落宿主机、`./volumes/milvus:/var/lib/milvus` 把 standalone 的索引文件落宿主机。三个容器各自是 ephemeral 的"代码 + 运行时载体",真正持久的东西全部在宿主机磁盘上,容器随时可以 `docker compose down && up` 重建,数据原地还在。这就是<b>镜像 = 不可变代码制品 / 容器 = ephemeral 运行实例 / volume = 持久化数据的家</b>的三分法,记住这个分工往后看任何 compose 文件就不会再有"为啥这个容器也挂卷"的疑问——只要它要存东西,它就得挂。

### 5.9 buildx push Hub + 资源观察

&emsp;&emsp;Dockerfile 写好、compose 写好,下一步就是 build 镜像 + push 到 Docker Hub。本机是 Mac M 系列(arm64),生产服务器是 Linux amd64——需要走 `docker buildx --platform linux/amd64` 跨架构 build。第四章我们走过一遍 buildx 的基础流程,这一节直接给文搜图的命令组合。

```bash
# 5.9 本机 buildx amd64 + push 到 Docker Hub
cd /path/to/text2image_search

# 登录 Docker Hub (第一次跑或 token 过期时)
docker login

# backend 镜像 (约 25-40 分钟, Mac M 系列上 QEMU 仿真 amd64)
docker buildx build \
    --platform linux/amd64 \
    --file backend/Dockerfile.gpu \
    --tag yanggggg/wensoutu-backend:0.1 \
    --push \
    backend/

# frontend 镜像 (约 2-3 分钟)
docker buildx build \
    --platform linux/amd64 \
    --tag yanggggg/wensoutu-frontend:0.1 \
    --push \
    frontend/
```

```powershell
# Windows PowerShell —— 命令一致, 路径用 Windows 风格
cd D:\projects\text2image_search

docker login
docker buildx build --platform linux/amd64 `
    --file backend/Dockerfile.gpu `
    --tag yanggggg/wensoutu-backend:0.1 `
    --push backend/
docker buildx build --platform linux/amd64 `
    --tag yanggggg/wensoutu-frontend:0.1 `
    --push frontend/
```

&emsp;&emsp;`docker buildx build --platform linux/amd64 --push` 一条龙:Mac M 系列启用 QEMU 仿真 amd64 指令集,build 完直接推到 Hub。backend ~5 GB 镜像走 QEMU 仿真约 25-40 分钟(QEMU 是跨架构固有代价),frontend ~80 MB 约 2-3 分钟。**只构 amd64 不构 arm64**——Mac 本机走 host venv,arm64 用不到,只构生产服务器需要的 amd64 一份。镜像分发闭环:本机 build → Hub → 服务器 `docker compose pull`,中间不经 scp / U 盘。build 完用 `docker stats` 看本机 Milvus 三件套的实际资源占用——这是日常运维第一眼工具。

> 提示:**buildx 跨次 build 经常不命中 cache,改一行源码也可能重跑 5 GB 依赖**。原因是 buildx 的 layer cache 不像普通 `docker build` 那样存在本地 docker daemon 里,而是放在 buildkit 容器内部——Docker Desktop 重启 / 长时间不用被 GC / builder 实例切换都会丢 cache。修法是用 `--cache-from` + `--cache-to` 把 cache 推到 Hub 作为外部存储:
>
> ```bash
> docker buildx build \
>     --platform linux/amd64 \
>     --tag yanggggg/wensoutu-backend:0.2 \
>     --cache-from type=registry,ref=yanggggg/wensoutu-backend:buildcache \
>     --cache-to type=registry,ref=yanggggg/wensoutu-backend:buildcache,mode=max \
>     --push \
>     -f backend/Dockerfile.gpu \
>     backend/
> ```
>
> 第一次跑:buildkit 把 layer cache 推到 Hub `:buildcache` tag(几 GB);**第二次跑:从 Hub 拉 `:buildcache` 比对 layer hash,依赖层 hash 没变就直接 skip(不重传)**,只重跑改过的源码层,28 分钟 → 3-5 分钟。代价是 Hub 多占一个 tag(`mode=max` 推所有中间层,`mode=min` 只推最终层省空间)。CI/CD 多人协作或多机 build 时,共享同一份 `:buildcache` tag 收益更大。

```bash
# 5.9 本机 docker stats 看 Milvus 三件套占用 (hybrid 状态)
docker stats --no-stream
```

&emsp;&emsp;`--no-stream` 给一次性快照(不带它会刷新模式持续打印,Ctrl+C 退出)。本机 hybrid 模式只跑三件套,实测输出大概是这样:

```text
NAME         CPU %    MEM USAGE / LIMIT      MEM %    NET I/O
t2i-milvus   0.10%    553.6MiB / 15.6GiB     3.46%    19.4MB / 41.5MB
t2i-minio    0.05%    117.9MiB / 15.6GiB     0.74%    1.62MB / 30.8kB
t2i-etcd     0.83%    30.3MiB  / 15.6GiB     0.19%    1.07MB / 1.59MB
```

&emsp;&emsp;三件套合计内存约 700 MiB——基础设施在内存上是低开销的。`LIMIT` 列在 Mac Docker Desktop 上显示 `15.6GiB`(我们设的 16 GB 配额减去 VM 开销,docker stats 输出本身用 GiB 计量),Linux 原生 docker 上会显示宿主机全部物理内存(下一章 4GPU24G 是 188 GiB),这是 hybrid 跟 Linux 生产环境资源观察的核心差异。

&emsp;&emsp;趁这节也提前在服务器侧看一眼 GPU 空载基线,把"空载稳态"先记在脑子里,上线后再看 `nvidia-smi` 才能识别哪些显存是 backend 容器吃掉的。

```bash
# 在 4GPU24G 服务器侧跑 (本机 ssh 上去)
ssh 4GPU24G 'nvidia-smi'
# 期望: 4× RTX 3090 全部空载, 显存占用 ~10 MiB / 24 GiB, 温度 39-41°C, 功耗 13-23 W
# 这是上线前的基线; 上线后 backend 容器加载双 2B 模型, GPU 0 显存升到 ~10-12 GiB
```

&emsp;&emsp;空载时四张卡每张显存只用 ~10 MiB(驱动自身的常驻),`docker compose up` 之后 backend 的 cu128 wheel 自动挑 GPU 0 加载双 2B 模型,显存升到 ~10-12 GiB——这条对照在 6.4 节验收清单里会再用到。

&emsp;&emsp;<b>第五章 → 第六章</b>:本机 hybrid 跑通了、5 服务生产编排写好了、5 GB backend + 80 MB frontend 镜像都推到了 Docker Hub。但镜像只在 Hub,真正"上线"还没发生——下一章把它拉到一台 Linux + NVIDIA GPU 生产服务器,起服务、配 mirror、接反代。

---

## <center>第六章 上 4GPU24G:从 mirror 到上线</center>

&emsp;&emsp;本章把镜像从 Hub 拉到企业自有 4GPU24G 服务器(Ubuntu 22.04 + Docker 29 + 4× RTX 3090 + nvidia CDI 已就绪),跑通"服务器审计 → mirror 配置 → 上传模型 → 上线发布 + 验收"完整闭环。换主角到 4GPU24G——一台企业自有 Linux 服务器,4× RTX 3090 + 188 GiB 内存 + 1.3 TB 磁盘,Ubuntu 22.04.5 + Docker 29.1.3 + Docker Compose v2.x + nvidia CDI(Container Device Interface,容器 GPU 透传新标准)都已装好。要做的事不多——把第五章 push 上 Hub 的镜像拉下来、编排文件 scp 过去、模型 rsync 进卷、`docker compose up -d` 一条命令,上线就完成。这台是企业内网服务器,frontend 默认只绑 `127.0.0.1:3000` 给本机调试.

&emsp;&emsp;但我们通常会遇到一个问题:<b>企业内网防火墙拦了 Docker Hub 直连</b>。第四章在腾讯云 CVM 上用了腾讯云内网 mirror(`mirror.ccs.tencentyun.com`),那是腾讯云自家资源、只在 CVM 内可达,4GPU24G 不在腾讯云,这个 mirror 用不了。好消息是国内有一批通用公网 docker mirror(daocloud / 1ms.run / dockerhub.icu)——它们是<b>全透明代理</b>,任何镜像(包括我们 push 的 `yanggggg/wensoutu-backend:0.1` 这种个人命名空间公开仓库)都能透明拉到。

&emsp;&emsp;本章把"两种 mirror 场景"的对照梳理清楚——云厂商内网 mirror(腾讯云 / 阿里云 / 华为云,各家自用,极快)对比通用公网 mirror(daocloud / 1ms.run / dockerhub.icu,任何 Linux / Mac 都用,稍慢但全场景)。我们以后无论在哪种环境部署,都知道该挑哪种 mirror。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/6.0-%E4%B8%8A%E7%BA%BF%E5%9B%9B%E6%AD%A5%E8%B5%B0-d1a14a66.png" width=80%>
</div>

> 注意:图里 backend 标的 `:3001` 是<b>容器内 FastAPI 监听端口</b>(`Dockerfile` 里 `EXPOSE 3001` 写死)。4GPU24G 上 host 侧 3001 被别的服务占用,本课用 `BACKEND_HOST_PORT=3011` 把 host 侧改到 3011(见 6.3 节 `.env.prod`),所以<b>从 4GPU24G 宿主机 curl 调试 backend 要用 `127.0.0.1:3011`</b>;<b>容器内服务(frontend / nginx 反代)仍走 `backend:3001`</b>(compose 内部网络按服务名解析,跟 host 映射无关)。这一层"容器内端口固定、host 端口变量化"的设计正是为了让同一份镜像跨机器复用——3001 在腾讯云 / 4GPU24G / 同事电脑上撞没撞别人,各自改 `.env` 一行即可,Dockerfile 不动。

### 6.1 服务器侧前置审计

&emsp;&emsp;接入 4GPU24G 第一件事是审计——这是 IT 提前装好的服务器,我们只做核对、不做从零安装。审计六件事:GPU 数和显存、Docker 版本、Compose 版本、GPU runtime 是否注册、磁盘余量、内存余量。下面这套命令是接手任何新 Linux GPU 服务器的标准"接机清单"。

```bash
# 6.1 服务器审计 (一次性, 在 4GPU24G 上跑)
ssh 4GPU24G

# GPU 数 + 显存 + 温度 + 功耗
nvidia-smi
# 期望: 4× NVIDIA GeForce RTX 3090, 每张 24 GiB 显存,
# 空载温度 39-41°C, 功耗 13-23 W

# Docker 版本
docker --version
# 期望: Docker version 29.1.3 (或更高 29.x)

# Compose 版本
docker compose version
# 期望: Docker Compose version v2.30+ (v2.x 系列)

# GPU runtime 是否注册 (Docker 29+ 内置 CDI 标准)
docker info 2>&1 | grep -iE "runtime|cdi" | head
# 期望看到 nvidia 出现在 Runtimes 列表, 以及 nvidia.com/gpu=0..3 全部四张卡已注册

# 根盘余量
df -h /
# 期望余量 ≥ 100 GiB (4GPU24G 实测剩余 1.3 TB)

# 内存余量
free -h
# 期望 RAM ≥ 32 GiB (4GPU24G 实测 188 GiB)
```

&emsp;&emsp;`docker info 2>&1 | grep -iE "runtime|cdi"` 是关键一步——查 nvidia GPU runtime 有没有被 Docker daemon 识别到,如果没注册容器拿不到 GPU(注意正则要用双引号包起来,否则 shell 把 `|` 解析成管道、`cdi` 当成命令名)。任何一项不达预期就先解决环境再上线。这套清单可复用——接手任何 Linux GPU 服务器,六条命令跑一遍,基线就清楚了。

> <font size=2>**【名词解释】<font color=red>CDI</font>**(Container Device Interface,容器设备接口)— NVIDIA + 容器社区合作的容器 GPU 透传标准,代替老式的 nvidia-container-toolkit。Docker 29+ 内置 CDI 支持,GPU 设备注册成 `nvidia.com/gpu=0/1/.../all` 形式,compose 用 `deploy.resources.reservations.devices` 声明即可绑卡。</font>

&emsp;&emsp;4GPU24G 实测的完整审计结果:4× NVIDIA GeForce RTX 3090(每张 24 GiB 显存)、Docker 29.1.3、Docker Compose v2.x(`docker compose version` 输出 v2.30+)、nvidia.com/gpu CDI 全部四张卡已注册、根盘剩余 1.3 TB(`/data/docker/` 当前占 9.3 GiB)、内存 188 GiB。**容纳文搜图相关资源绰绰有余**——5 GiB backend 镜像 + 80 MiB frontend 镜像 + 8 GiB 模型卷 + 70 MiB 演示图 + Milvus 数据卷,合计不到 20 GiB,占根盘剩余空间的 1.5%。

### 6.2 通用 docker mirror 配置

&emsp;&emsp;企业内网防火墙一般会拦截对 `registry-1.docker.io` 的直连。第四章我们在腾讯云 CVM 上遇到过类似情况,当时用的是腾讯云<b>内网 mirror</b>(`mirror.ccs.tencentyun.com`)——但那是腾讯云自家资源,只在腾讯云 CVM 内可达,4GPU24G 不在腾讯云用不了。

&emsp;&emsp;解决方法是配通用<b>公网 mirror</b>——国内有一批 docker registry 透明代理,任何 Linux / Mac 都可用,本课用三家:`https://docker.m.daocloud.io`、`https://docker.1ms.run`、`https://dockerhub.icu`(跟第二章 2.2 节本机 Docker Desktop 配过的同款,只是位置换到服务器侧,且服务器侧还要保留 IT 装机时配的 `data-root` / `runtimes` 字段)。改一份 `/etc/docker/daemon.json` 把三家加进去,daemon 重启就生效。下面这套命令覆盖"备份 → 写新配置 → 重启 → 验证"四步。

```bash
# 6.2 配 daemon mirror (服务器侧, 一次性)

# 第一步: 先备份现有 daemon.json (复用本套配置到其他机器时这步很重要,
# 因为目标机器可能已经有 data-root / runtimes 等字段, 不能整份覆盖)
sudo cp /etc/docker/daemon.json /etc/docker/daemon.json.bak 2>/dev/null \
    || echo "无已有配置, 跳过备份"

# 看现有配置 (如果没有就是空)
cat /etc/docker/daemon.json 2>/dev/null

# 第二步: 写新 daemon.json
# 下面这份是本课 4GPU24G 服务器的"完整参考样本"——已经包含 IT 装机时配的
# data-root / runtimes 字段, 加上我们这次要加的 registry-mirrors,
# 三段合在一起是 4GPU24G 上 daemon.json 的最终完整内容,所以可以整份 tee 写下去。
# 复用到其他机器时, 先 cat /etc/docker/daemon.json 看现有字段, 然后:
#   - 现有字段都不在下面 ⇒ 可以照样 tee 整份覆盖
#   - 现有字段下面没有 ⇒ 不能 tee, 用 jq 合并字段或手编辑 (见后注)
sudo tee /etc/docker/daemon.json > /dev/null <<'EOF'
{
    "data-root": "/data/docker",
    "runtimes": {
        "nvidia": {
            "args": [],
            "path": "nvidia-container-runtime"
        }
    },
    "registry-mirrors": [
        "https://docker.m.daocloud.io",
        "https://docker.1ms.run",
        "https://dockerhub.icu"
    ]
}
EOF

# 第三步: 重启 docker daemon 让新配置生效
sudo systemctl restart docker

# 第四步: 验证 mirror 已注册
docker info 2>&1 | grep -A 5 "Registry Mirrors"
# 期望看到三行: https://docker.m.daocloud.io / https://docker.1ms.run / https://dockerhub.icu
```

&emsp;&emsp;这套命令重写 daemon 配置:`data-root: /data/docker` 把镜像存储指到大盘(4GPU24G 上当前占 9.3 GiB,根盘剩 1.3 TB);`runtimes: nvidia` 是 IT 装机时已配的 GPU runtime,这里在新 daemon.json 里保留;`registry-mirrors` 是我们新增的核心配置,daemon 拉镜像时按列表顺序依次尝试,daocloud 慢自动换 1ms.run 或 dockerhub.icu——给三家是冗余。**复用注意**:如果目标机器 daemon.json 已有不在本课样本里的字段(`log-driver` / `storage-driver` 等),`tee` 整份覆盖会丢字段,这种场景用 `jq` 合并而非覆盖(`sudo jq '.["registry-mirrors"]=[...]' /etc/docker/daemon.json | sudo tee daemon.json.new && sudo mv daemon.json.new daemon.json`)。

> <font size=2>**【名词解释】<font color=red>Registry Mirror</font>**(Registry Mirror,镜像源代理)— Docker daemon 通过 `/etc/docker/daemon.json` 的 `registry-mirrors` 字段配置的镜像下载代理。daemon 拉镜像时优先走 mirror,命中本地缓存直接返回,没命中向上游 Hub 拉 + 缓存。<b>全透明</b>:用户写 `docker pull yanggggg/wensoutu-backend:0.1` 不需要改 image 名,daemon 自动透过 mirror 转发(前提是公开镜像;私有镜像需先 `docker login`)。</font>

&emsp;&emsp;**提示**:`daocloud` 这套是<b>全透明代理</b>——任何公开镜像名都自动从上游 Hub 拉 + 缓存,实测 frontend 80 MB 镜像 22 秒(约 3.6 MB/s),backend 5 GB 镜像 5-10 分钟。私有仓库(Hub 上设了 Private)mirror 走不通,需要 `docker login` 鉴权后再拉。

&emsp;&emsp;两种 mirror 场景的对照表,我们在第七章 7.4 节"两种 mirror 场景对照"会完整给出,这里先把判断标准捎一句:<b>机器在哪家云就用哪家内网 mirror,机器不在云上就用通用公网 mirror</b>。

### 6.3 上线:两条并列路径

&emsp;&emsp;mirror 配好了,镜像能拉了。但 backend 容器首次启动还要从 ModelScope 下 8 GiB 模型权重——按上一节说的,首次 `docker compose up` 会触发在线下载,大约 30-40 分钟才能 healthy。这段时间相当漫长,容器一直显示 `(unhealthy)`。所以本节给出<b>两条并列路径</b>:<b>路径 A 本机上传模型</b>(适合需要快速上线、不希望容器首启等 30-40 分钟下载的场景)在本机提前 rsync 模型到服务器卷,上线时 60 秒所有服务 healthy;<b>路径 B 容器自动下载</b>(零准备成本)直接 `docker compose up -d`,backend 容器触发 ModelScope 在线下载,30-40 分钟后所有服务 healthy。两条路径都跑得通,挑哪条看具体场景的时间预算。

> 提示:企业共用服务器常见踩坑——`docker compose up -d backend` 报 `failed to bind host port 127.0.0.1:3001/tcp: address already in use`,host 端口被别的进程占了。先 `sudo ss -tlnp \| grep 3001` 看是什么进程,<b>不属于自己的服务别瞎 kill</b>(可能是同事/IT 的应用)。修法是在 `.env.prod` 加一行 `BACKEND_HOST_PORT=3011`(或别的空闲端口)换 host 侧端口,<b>容器内 FastAPI 仍监听 3001</b>,frontend 走 docker 内网解析 `backend:3001` 也不受影响**,只是宿主机 `curl` 调试时要换成新端口。本课 4GPU24G 实测撞到 host 上一个 5 月初起的 node 服务占着 3001,改用 3011 解决。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/6.3-%E4%B8%8A%E7%BA%BF%E4%B8%A4%E6%9D%A1%E8%B7%AF%E5%BE%84-3d50e859.png" width=80%>
</div>

#### 6.3.1 容器内用户 UID 与挂载卷权限对齐

&emsp;&emsp;两条路径开始之前,先把服务器侧的挂载卷目录所有权改对。文搜图 backend Dockerfile 用的是 `RUN useradd --system app`(不指定 `--uid`),这种写法<b>系统分配的 UID 在 system 用户范围内</b>(Debian/Ubuntu 默认 999),跟第三章 ai-todo 镜像 `useradd --system --uid 1000` 显式指定 1000 不同。本机 Docker Desktop 通过 osxfs / gvisor 文件系统层做了权限映射不会撞,但到了 Linux 服务器上必撞——docker 自动创建的宿主机挂载目录归 root 所有,容器内 UID=999 的 app 用户没有写权限,启动直接报 `unable to open database file` 或 `Permission denied`。

&emsp;&emsp;一个前置的工作目录约定:企业 IT 给我们的 4GPU24G 账号是 `root`,ssh 进去默认 HOME 是 `/root`,但项目数据不放 root HOME——统一落在 `/home/XiaoYangWorkSpace/wensoutu/` 这个企业工作目录下,方便后续运维和企业内其他项目一起管理。下面所有命令都显式写绝对路径 `/home/XiaoYangWorkSpace/wensoutu/...`,不用 `~` 简写避免歧义。

```bash
# === 在本机 Mac 跑(先 ssh 进 4GPU24G)===
ssh 4GPU24G

# === 以下命令在 4GPU24G 上跑 ===
# 第一步: 先拉一次镜像 (让 docker daemon 知道这份镜像)
docker pull yanggggg/wensoutu-backend:0.1

# 第二步: 查容器内 app 用户的真实 UID/GID
# 文搜图 backend 实测是 999:999, 跟 ai-todo 的 1000:1000 是预期差异
docker run --rm yanggggg/wensoutu-backend:0.1 id app
# 期望输出: uid=999(app) gid=999(app) groups=999(app)

# 第三步: cd 到企业工作目录, 建好七个数据目录, 把所有权改成 999:999
cd /home/XiaoYangWorkSpace
sudo mkdir -p wensoutu/volumes/{etcd,minio,milvus,models,images,caption_cache,uploads}
sudo chown -R 999:999 wensoutu/volumes/
```

&emsp;&emsp;这套 `id` + `chown` 是部署到 Linux 服务器的通用动作——同一份镜像的 UID 取决于 Dockerfile 里 `useradd` 怎么写,本课实测 ai-todo=1000、文搜图 backend=999,这是预期差异,凭印象记数字会翻车,必须每个镜像都跑 `docker run --rm <镜像> id <用户>` 实测一次再 chown。

> 提示:`chown` 的<b>时序</b>是个隐蔽的踩坑点——上面这步 chown 跑在空目录上,只把目录所有权改对了。**接下来 6.3.2 节 rsync 传上来的 8 GiB 模型文件,所有权会保留你本机 Mac 的源端 UID(通常是 501 之类),不是 999**。容器内 app 用户(UID 999)启动时往 `.lock/` 子目录写锁文件会报 `[Errno 13] Permission denied: '/app/backend/data/models/.lock/qwen___Qwen3-VL-Embedding-2B'`,然后 fallback 让 transformers 当 HF 模型 ID 去 huggingface.co 找,加上离线模式就报 `LocalEntryNotFoundError`,启动直接 exit。修法是 <b>rsync 完之后再 chown 一次</b>——本课 6.3.2 节 rsync 命令组最后会补一行 `sudo chown -R 999:999 /home/XiaoYangWorkSpace/wensoutu/volumes/` 把新传上来的文件归属改对。本课 4GPU24G 实测就漏掉了这步,backend 启动失败两次才补上。

#### 6.3.2 路径 A:本机上传模型(rsync 编排 + 模型到卷)

&emsp;&emsp;路径 A 的核心思路是<b>把"数据"和"代码"分开传</b>——代码走 docker push / pull(镜像走 Hub),数据走 rsync(模型直接进服务器卷)。模型传完后服务器侧 `volumes/models/` 已经有完整 ModelScope 格式的模型目录,backend 容器看到卷有模型直接跳过下载,加载到 GPU 只用 10-20 秒。传文件之前先看 `.env.prod` 应包含的字段——这份文件不进 git(里头有 LLM 密钥),`compose.prod.yml` 里 `${DOCKER_HUB_USERNAME}` / `${IMAGE_TAG}` / `${OPENROUTER_API_KEY}` / `${OPENAI_BASE_URL}` / `${LLM_MODEL}` 五个变量需在 `.env.prod` 给出值:

```bash
# .env.prod 最小模板 (在本机文搜图项目根目录创建)
# 注意: 这份文件含密钥, 务必加进 .gitignore, 不能提交

# === 镜像源 ===
DOCKER_HUB_USERNAME=yanggggg            # 替换成你的 Hub 用户名
IMAGE_TAG=0.1                           # 当前要部署的镜像 tag

# === LLM 接入 (OpenAI 兼容协议, 我们用 OpenRouter) ===
# 跟前置课 ai-todo 同一个 OpenRouter key, 不用申请新的
OPENROUTER_API_KEY=sk-or-v1-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
OPENAI_BASE_URL=https://openrouter.ai/api/v1

# 模型 ID (OpenRouter 上 Qwen3-VL 235B Instruct 的标识)
# 选它的理由: 国产, 中国部署无地域问题, 多模态 (text+image) + tool calling 全能力
# 海外模型 (gemini / gpt-4o / claude) 在中国大陆区可能报 403 地域不可用, 工业部署优选国产
# 完整列表: https://openrouter.ai/models
LLM_MODEL=qwen/qwen3-vl-235b-a22b-instruct
```

&emsp;&emsp;`LLM_MODEL` 是 Agent 模式专用——文搜图主流程走项目自部署的 Qwen3-VL Embedding + Reranker 双模型,不依赖它。它必须同时满足:① 多模态(describe_image 工具要传图);② 支持 tool calling(不支持的会报 `404 - 未找到支持工具使用的端点`);③ 部署目标地区可用(海外模型在中国大陆区报 `403`,**企业部署最隐蔽的坑**)。`qwen3-vl-235b-a22b-instruct` 三条都过,本课实测最稳。`config.py` 优先读 `OPENROUTER_API_KEY`、fallback 到 `OPENAI_API_KEY`——`base_url + key + model` 三段式是 OpenAI 兼容协议工业标准。

```bash
# 6.3.2 路径 A 本机上传模型 (本机, 上线前几小时跑)
cd /path/to/text2image_search

# 第一步: 传编排文件 (几 KB, 秒级)
# 注: /home/XiaoYangWorkSpace/wensoutu/volumes/ 目录和 chown 在 6.3.1 节已经做好
scp docker-compose.prod.yml .env.prod 4GPU24G:/home/XiaoYangWorkSpace/wensoutu/

# 第二步: 传模型 (8 GiB, 企业内网约 80 秒 ~100 MB/s, 外网约 5-6 分钟 ~25 MB/s)
# rsync -a 保留权限 / 时间戳, -v 详细输出, -z 传输压缩
# --exclude 排除 macOS 的元数据垃圾文件 (.DS_Store / ._开头的资源叉),
# 这些文件 transformers / ModelScope 不会读, 但污染服务器卷, 工业部署一律排除
rsync -avz \
    --exclude='.DS_Store' --exclude='._*' \
    backend/data/models/ 4GPU24G:/home/XiaoYangWorkSpace/wensoutu/volumes/models/

# 第三步: 传 26 张图 (70 MB, 几秒)
rsync -avz \
    --exclude='.DS_Store' --exclude='._*' \
    backend/data/images/ 4GPU24G:/home/XiaoYangWorkSpace/wensoutu/volumes/images/

# 第四步: rsync 之后再 chown 一次, 把新传上来的文件归属改成 999:999
# (rsync -a 默认保留源端 Mac 的 UID, 不补这步 backend 容器写 .lock/ 时 Permission denied)
ssh 4GPU24G "sudo chown -R 999:999 /home/XiaoYangWorkSpace/wensoutu/volumes/"
```

&emsp;&emsp;<b>Windows 用户怎么传 8 GiB 模型</b>:原生 PowerShell 没有 `rsync`,标准 Git Bash 也不带,所以直接套上面命令会卡在第二步。推荐用 <b>WSL 跑 rsync</b>(把项目目录映射成 `/mnt/d/...` 即可),或者退化到 `scp -r` 全量传(没 `--exclude` 和增量,代价是每次重传):

```powershell
# Windows PowerShell - 推荐: WSL 跑 rsync (装了 WSL Ubuntu 后)
cd D:\projects\text2image_search
scp .\docker-compose.prod.yml .\.env.prod 4GPU24G:/home/XiaoYangWorkSpace/wensoutu/

wsl rsync -avz --exclude='.DS_Store' --exclude='._*' `
    /mnt/d/projects/text2image_search/backend/data/models/ `
    4GPU24G:/home/XiaoYangWorkSpace/wensoutu/volumes/models/

wsl rsync -avz --exclude='.DS_Store' --exclude='._*' `
    /mnt/d/projects/text2image_search/backend/data/images/ `
    4GPU24G:/home/XiaoYangWorkSpace/wensoutu/volumes/images/

# 退化方案: 没装 WSL, 用 scp -r 全量传 (没增量, 每次都是 8 GiB)
# scp -r backend\data\models 4GPU24G:/home/XiaoYangWorkSpace/wensoutu/volumes/

ssh 4GPU24G "sudo chown -R 999:999 /home/XiaoYangWorkSpace/wensoutu/volumes/"
```

```bash
# === 回到 macOS / Linux 主线 ===

# 第五步: 服务器侧 pull 镜像 + 拉起 + 等 healthy + 关闭 (留卷数据)
ssh 4GPU24G '
cd /home/XiaoYangWorkSpace/wensoutu
docker compose -f docker-compose.prod.yml --env-file .env.prod pull
docker compose -f docker-compose.prod.yml --env-file .env.prod up -d

# 模型已经在卷里, 60 秒所有服务 healthy
sleep 60
docker compose -f docker-compose.prod.yml --env-file .env.prod ps

# 跑一次端到端验证: backend 健康端点 + frontend nginx 入口
curl -sS http://localhost:3001/api/health
curl -sI http://localhost:3000/ | head -1

# 验证通过后关闭 (留卷数据)
docker compose -f docker-compose.prod.yml --env-file .env.prod down
'
```

&emsp;&emsp;路径 A 实现<b>模型卷状态的提前到位</b>:跑完之后服务器卷里有完整模型,6.4 节 `up -d` 时 backend 看到 `/app/backend/data/models/` 已有就跳过下载、加载 GPU 只用 10-20 秒(对比从零下载 30-40 分钟)。两条 rsync 加 `--exclude='.DS_Store' --exclude='._*'` 是 Mac 学员标配——这俩元数据 ModelScope 不读但污染卷目录,跨系统传文件标配;结尾用 `down` 而不是 `down -v`,保留 bind mount 数据,避免哪天换 named volume 翻车。

> 提示:rsync 的图片目录如果格式混合(`.png` + `.jpg` + `.jpeg` + `.webp`),`backend/core/retrieval.py` 在文搜图主路径 glob 时只匹配 `.png` 和 `.jpg`,**漏 `.jpeg` / `.webp` 等扩展名导致新图进不了文搜图召回池**。工程上建议统一图库为单一扩展名(如全部 `.png`),或者改 `retrieval.py` 让 glob 覆盖跟 SimpleDirectoryReader `required_exts` 一致的所有扩展名。这是同一份代码在不同数据集下的隐蔽行为差异,跟 UID 999/1000、bf16 在 MPS 不撞 CUDA 撞是同类。

#### 6.3.3 路径 B:容器自动下载(backend 首启从 ModelScope 在线下)

&emsp;&emsp;路径 B 不在本机准备任何模型文件,直接让 backend 容器从 ModelScope 在线下。代价是首次启动到 healthy 需要 30-40 分钟(取决于服务器到 ModelScope 的带宽),期间 `docker compose ps` 一直显示 `(unhealthy)`——这是预期行为,不是出错,`HEALTHCHECK` 的 `start_period=1800s` 就是给在线下载留的窗口。

```bash
# 6.3.3 路径 B 容器自动下载 (服务器侧)
# 把 .env.prod 和 docker-compose.prod.yml 先 scp 到服务器
# === 本机 ===
cd /path/to/text2image_search
scp docker-compose.prod.yml .env.prod 4GPU24G:/home/XiaoYangWorkSpace/wensoutu/

# === ssh 进 4GPU24G ===
ssh 4GPU24G
cd /home/XiaoYangWorkSpace/wensoutu

# 直接 pull 镜像 + 起容器, 模型由 backend 容器自己下载
docker compose -f docker-compose.prod.yml --env-file .env.prod pull
docker compose -f docker-compose.prod.yml --env-file .env.prod up -d

# 看下载进度: backend 容器会在 logs 里打 ModelScope 的下载行
docker compose -f docker-compose.prod.yml --env-file .env.prod logs -f backend

# 期望日志(节选):
#   Downloading [Qwen3-VL-Embedding-2B/model.safetensors]: 100% xx/xx ...
#   Downloading [Qwen3-VL-Reranker-2B/model.safetensors]: 100% xx/xx ...
#   [完成] 检索引擎和Agent就绪

# 期间 ps 看到 backend 一直 (unhealthy), 这是 start_period=1800s 窗口内的正常状态
docker compose -f docker-compose.prod.yml --env-file .env.prod ps

# 约 30-40 分钟后所有服务都到 healthy
```

&emsp;&emsp;成功标志:`logs -f backend` 看到 `[完成] 检索引擎和Agent就绪`,`ps` 列里所有服务变 `(healthy)`。跑完一次模型也在卷里,下次重启就跟路径 A 一样秒级加载——<b>本机不需要预先有 8 GiB 模型文件</b>,团队任意机器 ssh + scp 一份 compose 就能完成首次上线。两条路径选型简单:本机有模型选 A(60 秒 healthy),本机没模型选 B(30-40 分钟下载,首次慢但零准备)。

### 6.4 上线发布 + 验收清单

&emsp;&emsp;模型和镜像都到位后,正式上线只一条命令——`docker compose up -d`。秒级返回(因为镜像已经在服务器、模型已经在卷里、Milvus 三件套数据卷也跑过了)。但跑完命令不等于上线完成,我们要走一份<b>验收清单</b>,每一项对应一个具体的服务能力,六项全过才算上线成功。

```bash
# 6.4 上线发布 + 验收 (服务器侧, 每次发布)
ssh 4GPU24G
cd /home/XiaoYangWorkSpace/wensoutu

# 一条命令拉起全部
docker compose -f docker-compose.prod.yml --env-file .env.prod up -d

# 等约 60 秒 (路径 A 模型预先传过的话 backend 加载模型 10-20 秒)
sleep 60

# 验收六项, 每项必须通过

# ① 5 服务全部 Up + (healthy)
docker compose -f docker-compose.prod.yml --env-file .env.prod ps

# ② backend 日志看到 "[完成] 检索引擎和Agent就绪"(模型加载完成信号)
docker compose -f docker-compose.prod.yml --env-file .env.prod logs backend --tail 20

# ③ GPU 0 占约 10-12 GiB 显存 (双模型加载完稳态)
nvidia-smi

# ④ backend /api/health 返回 ok
curl -sS http://localhost:3001/api/health
# 期望: {"status":"ok","message":"Multimodal RAG Agent Backend Running"}

# ⑤ frontend nginx 80 端口返回 200
curl -sI http://localhost:3000/ | head -1
# 期望: HTTP/1.1 200 OK

# ⑥ 工位机 ssh tunnel 后浏览器开 http://localhost:3000/ 实际跑一次"红色的包"检索
#    本机另开终端: ssh -L 3000:localhost:3000 4GPU24G (保持窗口开着)
# 期望: N 张图片返回 + Reranker 精排分数显示
```

&emsp;&emsp;`docker compose up -d` 按 `depends_on` 拓扑序拉起 5 服务(etcd / minio 同时起 → standalone 等两者 healthy → backend 等 standalone healthy → frontend 等 backend started)。六项验收对应端到端能力的逐项确认——容器编排状态 / 模型加载信号 / GPU 显存 / 后端 API / 前端 nginx / 真实业务流程,少一项就不算上线。**清单可复用**:以后每次发新版本(更新镜像 tag → pull → up)都跑一遍这六项,通过才算上线,不通过立即回滚。

> 提示:即使按路径 A 把完整模型 rsync 进 `/home/XiaoYangWorkSpace/wensoutu/volumes/models/`,backend 容器启动时仍可能尝试调 ModelScope / HuggingFace 拿模型 metadata —— `transformers` 库的 `AutoModel.from_pretrained` 默认会先 online 校验版本,本地有文件也照样发 HEAD 请求。企业内网到 `huggingface.co` 通常没路由(实测报 `Failed to establish a new connection: [Errno 101] Network is unreachable`),每个 metadata 文件卡 5 次 retry 共 23 秒,十几个文件累计几分钟。彻底离线的修法是在 `docker-compose.prod.yml` 的 `backend.environment` 段加两行 `HF_HUB_OFFLINE: "1"` + `TRANSFORMERS_OFFLINE: "1"`,让 transformers 完全跳过 online 校验,直接读本地文件。**比起改 Dockerfile 重 build 镜像,在 compose 里注入环境变量更对**——运行时配置和镜像解耦,改完 `up -d --force-recreate backend` 几秒生效,不用走 build / push / pull 那一套。本课 4GPU24G 实测就是撞到这个,补完两个变量后 backend 启动从卡几分钟降到 10-20 秒。

&emsp;&emsp;内网部署上线闭环——服务在 4GPU24G 服务器跑起来了,工位机 ssh tunnel 后浏览器能开,文搜图可用。这台是企业内网服务器不对外公网,我们停在"compose up + 验收清单"这一步;以后要对外暴露 + HTTPS,套第四章 ai-todo 上腾讯云用过的"宿主机 nginx + certbot"那一套,把上游从"venv 进程端口"换成"容器映射端口"即可。下一章我们回顾产物和判断框架,把决策表与对照表一并收口。

---

## <center>第七章 课程回顾</center>

&emsp;&emsp;走到这里,容器化部署的完整链路我们已经全部跑过了——第一章认识 Docker 三对象(镜像 / 容器 / 仓库)、第二章 Docker Desktop 上手、第三章单容器 ai-todo 本机跑通、第四章跨架构 + 上 Hub + 腾讯云上线、第五章文搜图 compose 5 服务编排、第六章企业 4GPU24G 服务器 GPU 容器上线。这一章不是简单复习,是结构化收口——把六件产物清单、单 docker / compose / k8s 的决策表、两种 mirror 场景对照、容器派 vs 进程派对照、k8s 进阶方向五件事一次摆齐,让我们以后对类似项目能直接对照决策。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/container-deploy/2026-05-21/7.0-%E8%AF%BE%E7%A8%8B%E5%9B%9E%E9%A1%BE%E5%85%A8%E6%99%AF-97591be7.png" width=80%>
</div>

### 7.1 六件产物自检清单

&emsp;&emsp;学完本课我们应该带走六件能直接复用的产物,每一件都对应一个具体场景的工程能力。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>本课六件交付产物</font></p>
<div class="center">

| 产物 | 落点章节 | 复用场景 |
|---|---|---|
| ai-todo Dockerfile + buildx 跨架构 + 推 Hub | 第三章本机 Dockerfile + 第四章 buildx 跨架构 + 推 Hub | 任何 FastAPI / Python Web 应用要"打包 + 跨架构 + 上 Hub" |
| 文搜图完整 `docker-compose.prod.yml` + 本机 hybrid `docker-compose.deps.yml` | 5.8 完整 prod compose + 5.4 hybrid deps | 多服务 / 单机编排 / 5-10 服务场景 |
| backend GPU Dockerfile.gpu + frontend Dockerfile | 5.5 backend Dockerfile.gpu + 5.7 frontend Dockerfile | 任何带 PyTorch GPU 推理的服务 + 任何 SPA 静态前端 |
| 两种 mirror 场景对照表 + `daemon.json` 配置写法 | 6.2 通用 mirror 配置 + 7.3 两种 mirror 对照 | 任何 Linux 服务器 docker mirror 配置 |
| 4GPU24G 上线手册:审计 → mirror → 传模型 → up + 验收 | 6.1-6.4 上线四步走 | 任何企业自有 Linux GPU 服务器的上线流程 |
| 容器派 vs 前置课进程派对照表 | 7.4 容器派 vs 进程派对照 | 帮我们判断新项目走 docker 还是走 systemd 部署 |

</div>

&emsp;&emsp;六件产物对应本课从第三章到第六章共四章(第三、四、五、六章)的核心交付物——这不只是"学过什么"的清单,更是"以后遇到类似问题能直接拿出来用"的工具集。

### 7.2 单 docker / compose / k8s 决策表

&emsp;&emsp;Docker 生态有三档编排工具:单容器 `docker run`、`docker compose`、`Kubernetes`。本课主要覆盖前两档,第三档作为进阶方向给出关键词和入口工具。三档的适用场景如下表。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>单 docker / compose / k8s 决策表</font></p>
<div class="center">

| 工具 | 适用场景 | 典型项目 | 本课对应 |
|---|---|---|---|
| 单容器 `docker run` | 单服务 / 单机 / 不需要服务间协作 / 单进程对外暴露一个端口 | ai-todo / 单 API 服务 / 简单脚本 | 第三章 + 第四章 |
| `docker compose` | 多服务 / 单机 / 需要启动顺序 + 健康检查 + 服务间网络 / 一份 yaml 描述全部 / 服务数 ≤ 10 | 文搜图 5 服务 / RAG 应用 / 中小型微服务 | 第五章 + 第六章 |
| `Kubernetes` | 多机 / 弹性伸缩 / 滚动更新 / 灰度发布 / 多副本调度 / 服务数 > 10 或单服务多实例 | 大型生产系统 / SaaS 平台 | 进阶方向 |

</div>

&emsp;&emsp;判断标准是<b>服务数 × 部署机器数 × 流量级别</b>三个维度——本课在 1-5 服务 × 1 台机器 × 演示流量这个区间,`docker compose` 完全够用;真正到 10+ 服务、多台机器、需要灰度 / 滚动 / 弹性的场景,才需要往 k8s 走。

&emsp;&emsp;**提示**:`docker compose` 跟 `k8s` 不是替代关系,是不同复杂度档位的工具。如果手上是 5 个服务、跑在一台单机上,用 k8s 反而引入额外复杂度(需要装 etcd 集群、Master / Worker 节点、网络插件 CNI、存储插件 CSI,运维成本远高于 compose 的一份 yaml)。<b>选合适的工具档位</b>比"用最新的工具"更重要。

### 7.3 两种 mirror 场景对照

&emsp;&emsp;第四章 ai-todo 上腾讯云、第六章文搜图上 4GPU24G,我们用了两种不同的 docker mirror。这两种 mirror 的差异、适用场景、配置方式、性能特点,在下面这张表里完整摆开。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>云厂商内网 mirror vs 通用公网 mirror</font></p>
<div class="center">

| 维度 | 云厂商内网 mirror | 通用公网 mirror |
|---|---|---|
| 代表地址 | `mirror.ccs.tencentyun.com`(腾讯云)/ `https://<你的ID>.mirror.aliyuncs.com`(阿里云,需控制台申请)/ `swr-cache.cn-north-4.myhuaweicloud.com`(华为云) | `docker.m.daocloud.io` / `docker.1ms.run` / `dockerhub.icu` |
| 可达性 | 只在自家云的 CVM / ECS 内网可达 | 任何 Linux / Mac 都可用 |
| 速度 | 内网直连,GB 级镜像分钟级到位 | 公网带宽受限(实测 3.6 MB/s 起,因本地出口而异) |
| 适用机器 | 云厂商家的 CVM / ECS | 企业自有服务器 / 本地开发机 / 实验室主机 |
| 公开镜像透明拉 | 是，个人命名空间下的公开仓库也算 | 是，个人命名空间下的公开仓库也算 |
| 真私有仓库 | 仍需 `docker login` 鉴权 | 仍需 `docker login` 鉴权 |
| 本课对应章节 | 第四章 4.7 节"腾讯云内网 mirror" | 第六章 6.2 节"通用 docker mirror 配置" |

</div>

&emsp;&emsp;挑哪种的判断标准非常直接:<b>机器在哪家云就用哪家内网 mirror,机器不在云上就用通用公网 mirror</b>。两种 mirror 的配置方式都是改 `/etc/docker/daemon.json` 加 `registry-mirrors` 字段——这是 docker daemon 层级的全局配置,改完 `sudo systemctl restart docker` 重启生效,所有用户、所有 docker 客户端都走这套 mirror。

### 7.4 容器派 vs 进程派对照

&emsp;&emsp;同一个 ai-todo 项目,前置课走的是<b>进程派</b>(venv + systemd unit + nginx + 手动 scp + apt install),本课走的是<b>容器派</b>(Dockerfile + docker build + docker push + 服务器 docker pull + docker run + nginx 一行不改)。两派的差异集中落在五个维度——这张对照表的每一项都有前置课和本课两边的实证支撑。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>同一个 ai-todo,容器派 vs 进程派五维对照</font></p>
<div class="center">

| 维度 | 前置课:进程派 | 本课:容器派 |
|---|---|---|
| 换机器复刻成本 | 重装 Python + pip install + 改 systemd unit + 改 nginx,半天 | `docker pull` + 一行 `docker run` + nginx 一行不改,几分钟 |
| 依赖管理 | venv lock + `requirements.txt`,机器层 Python 版本必须对 | 封进镜像 layer,机器层不需要装 Python |
| 隔离边界 | 进程级,共用系统库 + 共用 `/etc` 配置 | cgroup + namespace 级,独立文件系统视图 / 独立网络 / 独立进程空间 |
| 入口工具 | 宿主机 nginx + systemd 守护 | 宿主机 nginx + dockerd 守护,nginx 一行不改,上游从 venv 进程换成容器端口 |
| 版本迭代 | `git pull` + `pip install -r requirements.txt` + `systemctl reload` | `docker compose pull` + `docker compose up -d`,秒级回滚 |

</div>

&emsp;&emsp;前置课和本课的所有命令都是实测跑过的,这张对照表的每一项都有实证支撑。**容器派不是要否定进程派**——前置课的工作没有白做,nginx + certbot + systemd 这三件事在容器化时代依然在用:本课的 nginx 配置跟前置课完全一致(只是上游换了)、certbot 申请证书的流程一字不变、宿主机 dockerd 依然由 systemd 守护(`systemctl status docker`)。<b>容器化只是把"装服务"换了表达,前置课的工程素养完整保留</b>。

### 7.5 k8s 进阶方向

&emsp;&emsp;compose 撑不住的边界在哪里?七个具体场景:① 多机协同(compose 只管单机 docker daemon,跨机调度做不了);② 弹性伸缩(根据负载自动加减副本数);③ 滚动更新(零停机发布新版本);④ 灰度发布(部分流量打到新版本验证);⑤ 多副本健康调度(单个副本挂了自动重新调度到健康节点);⑥ 服务发现 + 负载均衡(`Service` 抽象);⑦ 配置 / 机密管理(`ConfigMap` + `Secret`)。这些是 k8s 的能力,也是 compose 的能力边界。

&emsp;&emsp;从 compose 进 k8s 不需要从零开始——本课的 `docker-compose.prod.yml` 可以走 `kompose convert` 一键转换成 k8s 资源清单(`Deployment` + `Service` + `PersistentVolumeClaim`),这是进入 k8s 方向的入口工具。继续往 k8s 走时,本课的镜像和 compose 产物就是入手起点。

> <font size=2>**【名词解释】<font color=red>kompose</font>**(Kubernetes Compose,Compose-to-k8s 转换工具)— Kubernetes 官方维护的 docker-compose 到 k8s 资源转换工具,`kompose convert -f docker-compose.prod.yml` 一键生成 `Deployment` / `Service` / `PersistentVolumeClaim` 等 k8s 资源清单。本课不展开,是进入 k8s 方向的入口工具。</font>

> <font size=2>**【名词解释】<font color=red>distroless</font>**(distroless,无发行版基镜像)— Google 维护的"无发行版"基镜像,只含应用运行所需的最小依赖(没有 shell / 包管理器 / apt),镜像体积比 slim 还小、攻击面更小,生产环境推荐。本节我们用 `python:3.11-slim` 起步,生产环境可以进一步换 distroless。</font>

&emsp;&emsp;k8s 是 Docker 生态向"集群编排"演进的下一档,继续学习时可以从声明式资源模型(Pod / Deployment / Service / Ingress / ConfigMap / Secret)入手,Kubernetes 官方文档(kubernetes.io)按"概念 → 任务 → 教程"三层组织,沿着关键词查官方文档是最稳的路径。

&emsp;&emsp;本课从"docker 是什么"出发,沿着<b>从单容器到多容器</b>的递进走完:单容器 ai-todo → 跨架构 + 上 Hub + 腾讯云、多容器 compose 文搜图 5 服务、企业 4GPU24G 服务器 GPU 容器上线——完整闭环。把"装服务"换成"分发一份镜像"、把"一台机器"换成"任何机器",我们从裸机派站到了容器派,以后无论部署什么项目,单容器 vs 多容器的取舍和这张决策表都能直接拿出来用。